# Polinizadores em mixes de plantas de cobertura (v2: nome científico e grupo funcional)

Zona de transição Amazônia–Cerrado, 2025. Análise para a *Agriculture, Ecosystems & Environment*.

**O que muda em relação à v1**
- Os táxons vêm da coluna **Nome científico** da planilha (ex.: *Apis mellifera*, *Trigona pallens*, Apoidea morfotipo 1), com nomes em inglês e itálico nas figuras.
- Todas as análises também são feitas por **grupo funcional** (Polinizador × Visitante floral).
- Novas análises de diversidade usadas na literatura: cobertura amostral (Chao et al. 2014), rank-abundância, partição da diversidade beta (Baselga 2010, 2013), rede bipartida (Triquet et al. 2024) e estabilidade temporal (Loreau & de Mazancourt 2008).
- O modelo de abundância total é comparado com uma versão que trata a data como efeito aleatório (teste de sensibilidade).

**Como usar:** ambiente padrão (**Python**) > **Ambiente de execução > Executar tudo**. Autorize o Drive na primeira célula. Os pacotes de R levam alguns minutos para instalar.

**Saídas**
- Bases limpas em **02_Dados**; tabelas em **02_Dados/Resultados_tabelas_v2**.
- Figuras em **04_Figuras** (PDF vetorial, TIFF 1000 dpi e PNG de prévia).

## 1. Acesso ao Google Drive e download dos dados

In [ ]:
# IDs das pastas do artigo no Drive
PASTA_DADOS    = "1QHjG8ZeflxcuATVyhZMNITqf1RkD3Nb-"   # 02_Dados
PASTA_FIGURAS  = "1mL37Z2t2i1-O11CC-Am-mJFpmyfpkOGb"   # 04_Figuras
ARQUIVO_DADOS  = "Polinizadores_2025_polinizador_visitante_floral_14092026.xlsx"

from google.colab import auth
auth.authenticate_user()
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload
import io, os, glob, mimetypes

drive = build("drive", "v3")
KW = dict(supportsAllDrives=True)

def listar(pasta):
    q = f"'{pasta}' in parents and trashed = false"
    return drive.files().list(q=q, fields="files(id, name, mimeType, modifiedTime)",
                              includeItemsFromAllDrives=True, **KW).execute()["files"]

def baixar(pasta, nome, destino):
    arqs = [f for f in listar(pasta) if f["name"] == nome]
    if not arqs:
        raise FileNotFoundError(f"{nome} não está na pasta {pasta}")
    req = drive.files().get_media(fileId=arqs[0]["id"], **KW)
    with io.FileIO(destino, "wb") as fh:
        dl = MediaIoBaseDownload(fh, req)
        done = False
        while not done:
            _, done = dl.next_chunk()
    print("Baixado:", nome, "| modificado em", arqs[0]["modifiedTime"])

def subpasta(pai, nome):
    for f in listar(pai):
        if f["name"] == nome and f["mimeType"] == "application/vnd.google-apps.folder":
            return f["id"]
    meta = {"name": nome, "mimeType": "application/vnd.google-apps.folder", "parents": [pai]}
    return drive.files().create(body=meta, fields="id", **KW).execute()["id"]

def enviar(caminho, pasta):
    """Envia o arquivo para a pasta; se já existir um com o mesmo nome, substitui o conteúdo."""
    nome = os.path.basename(caminho)
    mime = mimetypes.guess_type(caminho)[0] or "application/octet-stream"
    media = MediaFileUpload(caminho, mimetype=mime, resumable=True)
    exist = [f for f in listar(pasta) if f["name"] == nome]
    if exist:
        drive.files().update(fileId=exist[0]["id"], media_body=media, **KW).execute()
    else:
        drive.files().create(body={"name": nome, "parents": [pasta]}, media_body=media, fields="id", **KW).execute()

os.makedirs("/content/dados", exist_ok=True)
print("Arquivos em 02_Dados:", [f["name"] for f in listar(PASTA_DADOS)])
baixar(PASTA_DADOS, ARQUIVO_DADOS, f"/content/dados/{ARQUIVO_DADOS}")

# Envio ao Drive sem estourar a cota da API do Colab (que é compartilhada por todos os usuários):
# os arquivos são copiados pelo Drive montado em /content/drive. A API só é usada para descobrir o
# caminho de cada pasta (poucos pedidos), com nova tentativa e espera crescente se o Drive recusar.
import time, shutil
from googleapiclient.errors import HttpError
from google.colab import drive as gdrive

def executar(req, tentativas=8):
    for k in range(tentativas):
        try:
            return req.execute()
        except HttpError as e:
            if e.resp.status in (403, 429, 500, 502, 503, 504) and k < tentativas - 1:
                espera = min(5 * 2 ** k, 120)
                print(f"  Drive ocupado ({e.resp.status}); nova tentativa em {espera} s")
                time.sleep(espera)
            else:
                raise

_cache_pastas = {}
def listar(pasta, atualizar=False):
    if atualizar or pasta not in _cache_pastas:
        q = f"'{pasta}' in parents and trashed = false"
        arqs, token = [], None
        while True:
            r = executar(drive.files().list(q=q, fields="nextPageToken, files(id, name, mimeType, modifiedTime)",
                                            pageSize=1000, pageToken=token, includeItemsFromAllDrives=True, **KW))
            arqs += r["files"]; token = r.get("nextPageToken")
            if not token: break
        _cache_pastas[pasta] = arqs
    return _cache_pastas[pasta]

_caminhos = {}
def caminho_local(pasta):
    """Caminho da pasta no Drive montado (ex.: /content/drive/MyDrive/.../04_Figuras)."""
    if pasta not in _caminhos:
        if not os.path.ismount("/content/drive"):
            gdrive.mount("/content/drive")
        partes, atual = [], pasta
        while True:
            f = executar(drive.files().get(fileId=atual, fields="id, name, parents", **KW))
            if not f.get("parents"): break          # chegou em "Meu Drive"
            partes.append(f["name"]); atual = f["parents"][0]
        _caminhos[pasta] = os.path.join("/content/drive/MyDrive", *reversed(partes))
    return _caminhos[pasta]

def enviar(caminho, pasta):
    """Copia o arquivo para a pasta (substitui se já existir um com o mesmo nome)."""
    nome = os.path.basename(caminho)
    destino = caminho_local(pasta)
    if os.path.isdir(destino):
        shutil.copy2(caminho, os.path.join(destino, nome))
        return
    # Pasta fora do "Meu Drive" (ex.: compartilhada): envio pela API, devagar
    mime = mimetypes.guess_type(caminho)[0] or "application/octet-stream"
    media = MediaFileUpload(caminho, mimetype=mime, resumable=False)
    exist = [f for f in listar(pasta) if f["name"] == nome]
    if exist:
        executar(drive.files().update(fileId=exist[0]["id"], media_body=media, **KW))
    else:
        novo = executar(drive.files().create(body={"name": nome, "parents": [pasta]}, media_body=media,
                                             fields="id, name, mimeType, modifiedTime", **KW))
        listar(pasta).append(novo)
    time.sleep(1)

# Ícones próprios dos visitantes (opcional): PNGs na pasta 04_Figuras/Icones_polinizadores
os.makedirs("/content/icons", exist_ok=True)
PASTA_ICONES = subpasta(PASTA_FIGURAS, "Icones_polinizadores")
for f in listar(PASTA_ICONES):
    if f["name"].lower().endswith(".png"):
        baixar(PASTA_ICONES, f["name"], "/content/icons/" + f["name"])

ARQUIVO_BIOMASSA = "cover_crop_biomass_per_plot_2025.csv"
baixar(PASTA_DADOS, ARQUIVO_BIOMASSA, f"/content/dados/{ARQUIVO_BIOMASSA}")

## 2. Instalação dos pacotes de R
Binários pré-compilados (Posit Package Manager). A Liberation Sans substitui a Arial com as mesmas métricas.

In [ ]:
!apt-get -qq install -y fonts-liberation > /dev/null
!fc-cache -f > /dev/null
try:
    import rpy2
except ImportError:
    !pip -q install rpy2
%load_ext rpy2.ipython

In [ ]:
%%R
codename <- system("lsb_release -cs", intern = TRUE)
options(repos = c(CRAN = sprintf("https://packagemanager.posit.co/cran/__linux__/%s/latest", codename)),
        HTTPUserAgent = sprintf("R/%s R (%s)", getRversion(),
                                paste(getRversion(), R.version["platform"], R.version["arch"], R.version["os"])))
pkgs <- c("readxl", "dplyr", "tidyr", "stringr", "ggplot2", "patchwork", "scales", "glmmTMB",
          "emmeans", "multcomp", "multcompView", "car", "performance", "vegan", "DHARMa",
          "indicspecies", "iNEXT", "bipartite", "tibble", "png")
falta <- setdiff(pkgs, rownames(installed.packages()))
if (length(falta)) install.packages(falta, Ncpus = 2)
cat("Pacotes faltando:", setdiff(pkgs, rownames(installed.packages())), "\n")

## Pacotes, tema e funções de exportação
Nomes em inglês dos mixes, grupos funcionais e táxons; rótulos em itálico (plotmath); paleta Okabe-Ito; exportação no padrão AGEE.

In [ ]:
%%R
ARQUIVO_DADOS <- "/content/dados/Polinizadores_2025_polinizador_visitante_floral_14092026.xlsx"
ARQUIVO_BIOMASSA <- "/content/dados/cover_crop_biomass_per_plot_2025.csv"
setwd("/content")
suppressPackageStartupMessages({
  library(readxl); library(dplyr); library(tidyr); library(stringr)
  library(ggplot2); library(patchwork); library(scales)
  library(glmmTMB); library(emmeans); library(multcomp); library(multcompView)
  library(car); library(performance); library(vegan)
})
select <- dplyr::select; filter <- dplyr::filter; recode <- dplyr::recode   # evita conflitos com MASS e car
set.seed(2025)
for (d in c("out/data", "out/tables", "out/figs")) dir.create(d, recursive = TRUE, showWarnings = FALSE)

# Mixes e grupos funcionais em inglês
mix_en <- c("Carbono" = "Carbon", "Multipropósito" = "Multipurpose", "Polinizador" = "Pollinator")
fg_en  <- c("Polinizador" = "Pollinators", "Visitante Floral" = "Other flower visitors")

# Táxons: coluna "Nome científico" da planilha, traduzida para o inglês
taxon_en <- c(
  "Apis mellifera"               = "Apis mellifera",
  "Bombus sp."                   = "Bombus sp.",
  "Xylocopa sp."                 = "Xylocopa sp.",
  "Trigona pallens"              = "Trigona pallens",
  "Apidae/Apoidea (morfotipo 1)" = "Apoidea morphotype 1",
  "Apidae/Apoidea (morfotipo 2)" = "Apoidea morphotype 2",
  "Halictidae - morfotipo verde" = "Halictidae (green morphotype)",
  "Syrphidae"                    = "Syrphidae",
  "Coleoptera"                   = "Coleoptera",
  "Lepidoptera"                  = "Lepidoptera",
  "Hymenoptera (morfotipo 1)"    = "Hymenoptera morphotype 1",
  "Hymenoptera (morfotipo 2)"    = "Hymenoptera morphotype 2",
  "Hymenoptera (morfotipo 3)"    = "Hymenoptera morphotype 3")

# Rótulos com gênero/espécie em itálico (plotmath), para eixos, legendas e painéis
taxon_expr <- c(
  "Apis mellifera"                = 'italic("Apis mellifera")',
  "Bombus sp."                    = 'italic("Bombus")~"sp."',
  "Xylocopa sp."                  = 'italic("Xylocopa")~"sp."',
  "Trigona pallens"               = 'italic("Trigona pallens")')
lab_taxon_chr <- function(x) ifelse(x %in% names(taxon_expr), taxon_expr[x], paste0('"', x, '"'))
lab_taxon <- function(x) parse(text = lab_taxon_chr(x))

# Paletas acessíveis para daltônicos: Okabe-Ito (mixes) e Paul Tol (grupos funcionais)
pal_mix <- c("Carbon" = "#E69F00", "Multipurpose" = "#0072B2", "Pollinator" = "#009E73")
pal_fg  <- c("Pollinators" = "#332288", "Other flower visitors" = "#CC6677")
lab_mix <- "Cover crop mix"

theme_agee <- theme_classic(base_size = 8, base_family = "Arial") +
  theme(axis.text = element_text(colour = "black", size = 7),
        strip.background = element_blank(), strip.text = element_text(face = "bold", size = 8),
        legend.key.size = unit(3, "mm"), legend.title = element_text(size = 7.5),
        legend.text = element_text(size = 7), panel.grid = element_blank())
theme_set(theme_agee)

# Exporta cada figura em PDF vetorial (preferido pela AGEE), TIFF 1000 dpi e PNG de prévia.
# Larguras Elsevier: 1 coluna = 90 mm, 1,5 coluna = 140 mm, página inteira = 190 mm.
save_fig <- function(p, name, width = 190, height = 110) {
  ggsave(file.path("out/figs", paste0(name, ".pdf")), p, width = width, height = height,
         units = "mm", device = cairo_pdf)
  ggsave(file.path("out/figs", paste0(name, ".tiff")), p, width = width, height = height,
         units = "mm", dpi = 1000, compression = "lzw")
  ggsave(file.path("out/figs", paste0(name, ".png")), p, width = width, height = height,
         units = "mm", dpi = 300)
  invisible(p)
}
save_tab <- function(x, name) write.csv(x, file.path("out/tables", paste0(name, ".csv")), row.names = FALSE)

# Letras a partir das médias do modelo (letras iguais = sem diferença); "a" vai para a MAIOR média
letras <- function(emm) cld(emm, Letters = letters, adjust = "sidak", reversed = TRUE) %>% as.data.frame() %>%
  mutate(.group = str_trim(.group))

# Ajusta binomial negativa NB1 e NB2 e fica com a de menor AIC; se nenhuma convergir com a parcela
# como efeito aleatório (variância ~0), usa o modelo equivalente sem ela
fit_nb <- function(fixed, data, re = "(1 | plot_id)", families = list(nbinom1 = nbinom1, nbinom2 = nbinom2), disp = ~1) {
  for (fo in c(paste(fixed, "+", re), fixed)) {
    fits <- lapply(families, function(fm) {
      m <- suppressWarnings(try(glmmTMB(as.formula(fo), family = fm, dispformula = disp, data = data), silent = TRUE))
      if (!inherits(m, "try-error") && isTRUE(m$sdr$pdHess)) m else NULL
    })
    fits <- Filter(Negate(is.null), fits)
    if (length(fits)) return(fits[[which.min(sapply(fits, AIC))]])
  }
  NULL
}
cat("Configuração pronta.\n")

## Leitura e limpeza (nome científico e grupo funcional)
- Lê a planilha usando as colunas **Nome científico** e **Grupo funcional**.
- Corrige "20258" nas datas e calcula os **dias após a semeadura** (19/07/2025).
- Junta borboletas e mariposas em Lepidoptera e remove *Bombus* (zero indivíduos).
- Gera três bases: total por observação, por grupo funcional e por táxon.

In [ ]:
%%R
raw <- read_excel(ARQUIVO_DADOS, sheet = "DADOS", skip = 3, col_types = "text")
names(raw) <- c("date_raw", "block", "plot", "mix_pt", "fg_pt", "taxon_pt", "group_pt",
                "count", "wind", "temp", "hum", "lux")

# Datas: o Excel guarda número de série; uma célula foi digitada como "13/09/20258".
parse_date <- function(x) {
  x <- str_replace(x, "20258$", "2025")
  out <- as.Date(suppressWarnings(as.numeric(x)), origin = "1899-12-30")
  out[is.na(out)] <- as.Date(x[is.na(out)], format = "%d/%m/%Y")
  out
}
SOWING <- as.Date("2025-07-19")   # semeadura dos mixes

dat0 <- raw %>%
  mutate(date    = parse_date(date_raw),
         across(c(block, plot, count, wind, temp, hum, lux), as.numeric),
         mix     = factor(mix_en[mix_pt], levels = c("Carbon", "Multipurpose", "Pollinator")),
         fg      = factor(fg_en[fg_pt], levels = c("Pollinators", "Other flower visitors")),
         taxon   = unname(taxon_en[taxon_pt]),
         block   = factor(block),
         plot_id = factor(paste0("B", block, "_P", plot)))
stopifnot(!anyNA(dat0$date), !anyNA(dat0$mix), !anyNA(dat0$fg), !anyNA(dat0$taxon), !anyNA(dat0$count))

# Avaliação de 01/10/2025 (74 dias após a semeadura) excluída: fim da floração, com quase nenhuma visita
# (2,3 visitas por observação, contra 11 a 28 nas outras datas). Mantida, ela mascarava o efeito do vento,
# porque foi também o dia mais calmo. Para incluí-la de novo, use DATAS_EXCLUIDAS <- as.Date(character(0)).
DATAS_EXCLUIDAS <- as.Date("2025-10-01")
cat("Avaliações excluídas:", format(DATAS_EXCLUIDAS), "| linhas removidas:", sum(dat0$date %in% DATAS_EXCLUIDAS), "\n")
dat0 <- dat0 %>% filter(!date %in% DATAS_EXCLUIDAS)

# Borboletas e mariposas têm o mesmo nome científico (Lepidoptera): somamos as duas linhas
dat <- dat0 %>%
  group_by(date, block, plot, plot_id, mix, fg, taxon, wind, temp, hum, lux) %>%
  summarise(count = sum(count), .groups = "drop")

dates <- sort(unique(dat$date))
dat <- dat %>% mutate(eval = match(date, dates), day = as.numeric(date - min(dates)),
                      das = as.numeric(date - SOWING))

# Táxons sem nenhum registro (Bombus) saem das análises
zero_taxa <- dat %>% group_by(taxon) %>% summarise(n = sum(count)) %>% filter(n == 0) %>% pull(taxon)
dat <- dat %>% filter(!taxon %in% zero_taxa)
cat("Datas:", n_distinct(dat$date), "| Parcelas:", n_distinct(dat$plot_id), "| Táxons:", n_distinct(dat$taxon),
    "| Indivíduos:", sum(dat$count), "| Removidos (zero):", paste(zero_taxa, collapse = ", "), "\n")
print(dat %>% distinct(fg, taxon) %>% arrange(fg, taxon), n = 20)

# Base por observação (data × parcela) e por observação × grupo funcional
obs <- dat %>%
  group_by(date, eval, day, das, block, plot, plot_id, mix) %>%
  summarise(abund = sum(count), richness = sum(count > 0),
            wind = first(wind), temp = first(temp), hum = first(hum), lux = first(lux), .groups = "drop") %>%
  mutate(date_f = factor(eval), temp_s = as.numeric(scale(temp)), hum_s = as.numeric(scale(hum)),
         wind_s = as.numeric(scale(wind)), lux_s = as.numeric(scale(lux)), day_s = as.numeric(scale(day)))
clim_cols <- obs %>% select(date, plot_id, date_f, temp_s, hum_s, wind_s, lux_s, day_s)
obs_fg <- dat %>% group_by(date, eval, das, block, plot_id, mix, fg) %>%
  summarise(abund = sum(count), .groups = "drop") %>% left_join(clim_cols, by = c("date", "plot_id"))
obs_tx <- dat %>% left_join(clim_cols, by = c("date", "plot_id"))

write.csv(dat, "out/data/pollinators_clean_long_taxa.csv", row.names = FALSE)
write.csv(obs, "out/data/pollinators_clean_by_observation.csv", row.names = FALSE)
write.csv(obs_fg, "out/data/pollinators_clean_by_functional_group.csv", row.names = FALSE)
cat("Bases limpas salvas em out/data.\n")

## Tabela descritiva (táxon × mix × grupo funcional)
Tabela táxon × mix com o grupo funcional e a participação (%) de polinizadores e outros visitantes em cada mix.

In [ ]:
%%R
tab_tx <- dat %>% group_by(fg, taxon, mix) %>% summarise(n = sum(count), .groups = "drop") %>%
  pivot_wider(names_from = mix, values_from = n) %>%
  mutate(Total = Carbon + Multipurpose + Pollinator, Percent = round(100 * Total / sum(Total), 1)) %>%
  arrange(fg, desc(Total))
print(tab_tx, n = 20); save_tab(tab_tx, "T1_abundance_taxon_mix")

tab_fg <- dat %>% group_by(mix, fg) %>% summarise(n = sum(count), .groups = "drop") %>%
  group_by(mix) %>% mutate(percent = round(100 * n / sum(n), 1)) %>% ungroup()
print(tab_fg); save_tab(tab_fg, "T2_abundance_functional_group_mix")

## H1/H2 – Abundância total: modelo principal e teste de sensibilidade
- Modelo principal: binomial negativa (NB1 ou NB2 pelo AIC), mix + temperatura + vento + luminosidade + tendência sazonal + bloco, com a parcela aleatória.
- A umidade fica de fora por ter r = −0,96 com a temperatura.
- **Sensibilidade:** a mesma estrutura com a data como efeito aleatório. Se o efeito do mix se mantém nos dois, o resultado é robusto.

In [ ]:
%%R
# Modelo principal: binomial negativa com microclima, tendência sazonal, bloco e parcela
m_tot <- fit_nb("abund ~ mix + temp_s + wind_s + lux_s + poly(day_s, 2) + block", obs)
cat("Família:", family(m_tot)$family, "\n")
an_tot <- as.data.frame(Anova(m_tot, type = 2)); an_tot$term <- rownames(an_tot); print(an_tot)
save_tab(an_tot, "T3_anova_total_abundance")

# Sensibilidade: a data entra como efeito aleatório e absorve a variação entre avaliações
# (fenologia e clima do dia). Se o efeito do clima some aqui, ele atua entre datas, não dentro delas.
m_tot_d <- glmmTMB(abund ~ mix + temp_s + wind_s + lux_s + block + (1 | date_f) + (1 | plot_id),
                   family = family(m_tot), data = obs)
an_tot_d <- as.data.frame(Anova(m_tot_d, type = 2)); an_tot_d$term <- rownames(an_tot_d); print(an_tot_d)
save_tab(an_tot_d, "T4_anova_total_abundance_date_random")

emm_tot <- letras(emmeans(m_tot, ~ mix, type = "response")); print(emm_tot)
save_tab(emm_tot, "T5_emmeans_total_abundance")

# Razões de taxa (IRR) com IC 95% para o gráfico de efeitos
irr_tab <- function(m, model_name) {
  co <- summary(m)$coefficients$cond
  data.frame(model = model_name, term = rownames(co), IRR = exp(co[, 1]),
             low = exp(co[, 1] - 1.96 * co[, 2]), high = exp(co[, 1] + 1.96 * co[, 2]), p = co[, 4])
}
irr_all <- bind_rows(irr_tab(m_tot, "Seasonal trend"), irr_tab(m_tot_d, "Date as random effect"))
save_tab(irr_all, "T6_incidence_rate_ratios")

## Diagnóstico dos resíduos (DHARMa)
Diagnóstico dos resíduos dos dois modelos. O esperado é p > 0,05 nos testes de uniformidade, dispersão e zeros.

In [ ]:
%%R -w 190 -h 90 -u mm -r 150
library(DHARMa)
for (k in 1:2) {
  m <- list(m_tot, m_tot_d)[[k]]
  sim <- simulateResiduals(m, n = 1000)
  plot(sim)
  if (k == 1) {   # resíduos do modelo principal salvos para o material suplementar
    png("out/figs/Fig_DHARMa_residuals.png", width = 190, height = 90, units = "mm", res = 300); plot(sim); dev.off()
  }
  print(testDispersion(sim)); print(testZeroInflation(sim))
}

## Abundância por grupo funcional (polinizadores × outros visitantes)
- Modelo mix × grupo funcional (a interação diz se o mix afeta polinizadores e outros visitantes de forma diferente).
- Proporção de polinizadores entre todas as visitas, por um modelo beta-binomial.

In [ ]:
%%R
# Dispersão separada por grupo (dispformula = ~ fg): com uma dispersão só, a variância dos polinizadores
# (contagens altas) mascarava a diferença entre mixes nos outros visitantes.
m_fg  <- fit_nb("abund ~ mix * fg + temp_s + wind_s + lux_s + poly(day_s, 2) + block", obs_fg,
                re = "(1 | plot_id) + (1 | plot_id:fg)", disp = ~ fg)
fam_fg <- list(family(m_fg)); names(fam_fg) <- family(m_fg)$family
m_fg0 <- fit_nb("abund ~ mix + fg + temp_s + wind_s + lux_s + poly(day_s, 2) + block", obs_fg,
                re = "(1 | plot_id) + (1 | plot_id:fg)", families = fam_fg, disp = ~ fg)
print(anova(m_fg0, m_fg))
an_fg <- as.data.frame(Anova(m_fg, type = 2)); an_fg$term <- rownames(an_fg); print(an_fg)
save_tab(an_fg, "T7_anova_functional_group")
emm_fg <- letras(emmeans(m_fg, ~ mix | fg, type = "response")); print(emm_fg)
save_tab(emm_fg, "T8_emmeans_functional_group")

# Proporção de polinizadores entre os visitantes (binomial com parcela aleatória)
prop_fg <- obs_fg %>% select(date, eval, das, block, plot_id, mix, fg, abund, temp_s, wind_s, lux_s, day_s) %>%
  pivot_wider(names_from = fg, values_from = abund) %>%
  filter(Pollinators + `Other flower visitors` > 0)
m_prop <- glmmTMB(cbind(Pollinators, `Other flower visitors`) ~ mix + temp_s + wind_s + lux_s + poly(day_s, 2) +
                    block + (1 | plot_id), family = betabinomial, data = prop_fg)
print(Anova(m_prop, type = 2))
emm_prop <- letras(emmeans(m_prop, ~ mix, type = "response")); print(emm_prop)
save_tab(emm_prop, "T9_emmeans_proportion_pollinators")

## Modelos por táxon (nome científico)
Um modelo por táxon, como em Mallinger et al. (2019). Entram os táxons com pelo menos 100 indivíduos, registrados em 5 ou mais avaliações e em todos os blocos. Apoidea morfotipo 2 fica de fora: foi registrado em só 4 datas e não apareceu no bloco 4.

In [ ]:
%%R
# Critério: ≥ 100 indivíduos, registros em ≥ 5 das 12 avaliações e em todos os blocos
# (táxons concentrados em poucas datas geram modelos instáveis)
tx_info <- dat %>% group_by(taxon) %>%
  summarise(n = sum(count), n_evals = n_distinct(eval[count > 0]), n_blocks = n_distinct(block[count > 0])) %>%
  arrange(desc(n))
print(tx_info); save_tab(tx_info, "T10a_taxon_occurrence")
main_taxa <- tx_info %>% filter(n >= 100, n_evals >= 5, n_blocks == nlevels(dat$block)) %>% pull(taxon)
cat("Táxons modelados:", paste(main_taxa, collapse = ", "), "\n")
cat("Excluídos (poucas datas ou ausentes em algum bloco):",
    paste(setdiff(tx_info$taxon[tx_info$n >= 100], main_taxa), collapse = ", "), "\n")
tx_anova <- list(); tx_emm <- list()
for (tx in main_taxa) {
  mg <- fit_nb("count ~ mix + temp_s + wind_s + lux_s + poly(day_s, 2) + block", filter(obs_tx, taxon == tx))
  if (is.null(mg)) { cat("Modelo não convergiu:", tx, "\n"); next }
  a <- as.data.frame(Anova(mg, type = 2)); a$term <- rownames(a); rownames(a) <- NULL; a$taxon <- tx
  tx_anova[[tx]] <- a
  tx_emm[[tx]] <- letras(emmeans(mg, ~ mix, type = "response")) %>% mutate(taxon = tx)
}
tx_anova <- bind_rows(tx_anova); tx_emm <- bind_rows(tx_emm)
print(tx_anova %>% filter(term == "mix") %>% select(taxon, Chisq, Df, `Pr(>Chisq)`))
save_tab(tx_anova, "T10_anova_by_taxon"); save_tab(tx_emm, "T11_emmeans_by_taxon")

## H3 – Composição (PERMANOVA, betadisper, NMDS) e IndVal
- PERMANOVA (Bray-Curtis, Hellinger) com permutações dentro do bloco, para todos os táxons e separada por grupo funcional.
- Comparações par a par (Holm), betadisper e NMDS com os vetores dos táxons (envfit).

In [ ]:
%%R
comm <- dat %>% group_by(plot_id, block, mix, taxon) %>% summarise(n = sum(count), .groups = "drop") %>%
  pivot_wider(names_from = taxon, values_from = n, values_fill = 0)
env <- comm %>% select(plot_id, block, mix) %>% as.data.frame()
mat <- comm %>% select(-plot_id, -block, -mix) %>% as.matrix(); rownames(mat) <- env$plot_id
mat_h <- decostand(mat, "hellinger"); d_bc <- vegdist(mat_h, "bray")

pmv <- adonis2(d_bc ~ mix, data = env, permutations = how(nperm = 9999, blocks = env$block))
print(pmv); save_tab(cbind(term = rownames(pmv), as.data.frame(pmv)), "T12_permanova_all_taxa")

# PERMANOVA só com polinizadores e só com outros visitantes
for (g in levels(dat$fg)) {
  tx_g <- intersect(colnames(mat), unique(dat$taxon[dat$fg == g]))
  sub <- mat[, tx_g, drop = FALSE]; keep <- rowSums(sub) > 0
  r <- adonis2(vegdist(decostand(sub[keep, ], "hellinger"), "bray") ~ mix, data = env[keep, ],
               permutations = how(nperm = 9999, blocks = env$block[keep]))
  cat("\nPERMANOVA –", g, "\n"); print(r)
  save_tab(cbind(term = rownames(r), as.data.frame(r)), paste0("T13_permanova_", gsub(" ", "_", g)))
}

pairs_mix <- combn(levels(env$mix), 2, simplify = FALSE)
pw <- bind_rows(lapply(pairs_mix, function(pp) {
  i <- env$mix %in% pp; e <- droplevels(env[i, ])
  r <- adonis2(vegdist(mat_h[i, ], "bray") ~ mix, data = e, permutations = how(nperm = 9999, blocks = e$block))
  data.frame(pair = paste(pp, collapse = " vs "), R2 = r$R2[1], F = r$F[1], p = r$`Pr(>F)`[1])
}))
pw$p_holm <- p.adjust(pw$p, "holm"); print(pw); save_tab(pw, "T14_permanova_pairwise")
print(permutest(betadisper(d_bc, env$mix), permutations = 9999))

nmds <- metaMDS(mat_h, distance = "bray", k = 2, trymax = 200, autotransform = FALSE, trace = 0)
cat("Stress do NMDS:", round(nmds$stress, 3), "\n")
sc_site <- cbind(as.data.frame(scores(nmds, "sites")), env)
fit <- envfit(nmds, mat_h, permutations = 999)
sc_sp <- as.data.frame(scores(fit, "vectors")) * ordiArrowMul(fit)
sc_sp$taxon <- rownames(sc_sp); sc_sp$p <- fit$vectors$pvals
save_tab(sc_sp, "T15_nmds_taxon_vectors")

## Táxons indicadores (IndVal.g, 9999 permutações dentro do bloco)
Táxons indicadores de cada mix (De Cáceres & Legendre 2009).

In [ ]:
%%R
library(indicspecies)
iv <- multipatt(as.data.frame(mat), env$mix, func = "IndVal.g", duleg = TRUE,
                control = how(nperm = 9999, blocks = env$block))
summary(iv, alpha = 1, indvalcomp = TRUE)
iv_tab <- data.frame(taxon = rownames(iv$sign), iv$sign, row.names = NULL)
iv_tab$p_holm <- p.adjust(iv_tab$p.value, "holm")
save_tab(iv_tab, "T16_indval")

## H4 – Diversidade alfa por parcela (Hill, Shannon, Simpson, Pielou, Berger-Parker)
Números de Hill (q = 0, 1, 2), Shannon, Simpson, equabilidade de Pielou e dominância de Berger-Parker por parcela, com bloco no modelo.

In [ ]:
%%R
hill <- function(x, q) {
  p <- x[x > 0] / sum(x)
  if (q == 0) length(p) else if (q == 1) exp(-sum(p * log(p))) else 1 / sum(p^2)
}
div <- data.frame(env, N = rowSums(mat),
                  q0 = apply(mat, 1, hill, q = 0), q1 = apply(mat, 1, hill, q = 1),
                  q2 = apply(mat, 1, hill, q = 2)) %>%
  mutate(shannon = log(q1), simpson = 1 - 1 / q2, pielou = shannon / log(q0),
         berger_parker = apply(mat, 1, function(x) max(x) / sum(x)))
div_mean <- div %>% group_by(mix) %>%
  summarise(across(c(N, q0, q1, q2, shannon, simpson, pielou, berger_parker), list(mean = mean, se = ~ sd(.) / sqrt(n()))))
print(as.data.frame(div_mean)); save_tab(div, "T17_alpha_diversity_by_plot")

div_tests <- bind_rows(lapply(c("q0", "q1", "q2", "shannon", "simpson", "pielou", "berger_parker"), function(v) {
  m <- lm(as.formula(paste(v, "~ mix + block")), data = div)
  a <- as.data.frame(Anova(m, type = 2)); a$term <- rownames(a); rownames(a) <- NULL; a$response <- v
  e <- letras(emmeans(m, ~ mix)) %>% mutate(response = v)
  list(a = a, e = e)
}) %>% lapply(function(z) left_join(z$e, z$a %>% filter(term == "mix") %>% select(response, F = `F value`, p = `Pr(>F)`),
                                   by = "response")))
print(div_tests %>% select(response, mix, emmean, SE, .group, F, p))
save_tab(div_tests, "T18_alpha_diversity_tests")

## Diversidade por grupo: polinizadores, outros visitantes e todos os visitantes juntos
- Os mesmos índices por parcela calculados três vezes: com **todos os visitantes juntos**, só com **polinizadores** e só com **outros visitantes florais**.
- Cada índice é comparado entre os mixes (modelo linear com bloco; letras de Šidák, a = maior média).
- Resultado vai para a figura de barras `Fig_diversity_by_group` e para a tabela T43.

In [ ]:
%%R
# Mesmos índices por parcela, calculados com todos os táxons, só com polinizadores e só com outros visitantes florais.
# Teste: índice ~ mix + bloco (ANOVA tipo II); letras de Šidák com "a" para a maior média.
grupos_div <- list("All flower visitors" = colnames(mat),
                   "Pollinators" = intersect(colnames(mat), unique(dat$taxon[dat$fg == "Pollinators"])),
                   "Other flower visitors" = intersect(colnames(mat), unique(dat$taxon[dat$fg == "Other flower visitors"])))
div_fg <- bind_rows(lapply(names(grupos_div), function(g) {
  m <- mat[, grupos_div[[g]], drop = FALSE]
  data.frame(env, group = g, N = rowSums(m),
             q0 = apply(m, 1, function(x) if (sum(x) == 0) 0 else hill(x, 0)),
             q1 = apply(m, 1, function(x) if (sum(x) == 0) NA else hill(x, 1)),
             q2 = apply(m, 1, function(x) if (sum(x) == 0) NA else hill(x, 2)))
})) %>%
  mutate(shannon = log(q1), pielou = ifelse(q0 > 1, shannon / log(q0), NA),
         group = factor(group, levels = names(grupos_div)))
idx_fg <- c(N = "Abundance (visits)", q0 = "Richness (q = 0)", q1 = "Shannon diversity (q = 1)",
            q2 = "Simpson diversity (q = 2)", pielou = "Pielou evenness (J')")
div_fg_tests <- bind_rows(lapply(names(grupos_div), function(g) bind_rows(lapply(names(idx_fg), function(v) {
  d <- div_fg %>% filter(group == g, !is.na(.data[[v]]))
  m <- lm(as.formula(paste(v, "~ mix + block")), data = d)
  a <- as.data.frame(Anova(m, type = 2))
  letras(emmeans(m, ~ mix)) %>%
    left_join(d %>% group_by(mix) %>% summarise(mean = mean(.data[[v]]), se = sd(.data[[v]]) / sqrt(n()),
                                                 n_plots = n(), .groups = "drop"), by = "mix") %>%
    mutate(group = g, response = v, F = a["mix", "F value"], df = a["mix", "Df"],
           df_res = a["Residuals", "Df"], p = a["mix", "Pr(>F)"])
})))) %>% mutate(group = factor(group, levels = names(grupos_div)), response = factor(response, levels = names(idx_fg)))
print(div_fg_tests %>% select(group, response, mix, mean, se, .group, F, p) %>% arrange(group, response, mix))
save_tab(div_fg, "T42_alpha_diversity_by_group_plot")
save_tab(div_fg_tests %>% select(group, response, mix, n_plots, mean, se, emmean, SE, .group, F, df, df_res, p),
         "T43_alpha_diversity_by_group_tests")

## Rarefação por tamanho amostral e por cobertura (iNEXT; Chao et al. 2014)
Rarefação e extrapolação por tamanho amostral e por **cobertura amostral** (Chao et al. 2014). A comparação na mesma cobertura é a mais justa quando o número de indivíduos difere entre mixes.

In [ ]:
%%R
library(iNEXT)
abund_list <- lapply(split(as.data.frame(mat), env$mix), function(x) { s <- colSums(x); s[s > 0] })
inx <- iNEXT(abund_list, q = c(0, 1, 2), datatype = "abundance", endpoint = 2 * max(sapply(abund_list, sum)))
print(inx$DataInfo); print(inx$AsyEst)
save_tab(inx$AsyEst, "T19_inext_asymptotic"); save_tab(inx$DataInfo, "T20_inext_sample_coverage")
# Comparação na mesma cobertura amostral (a menor cobertura observada entre os mixes)
cov_min <- min(inx$DataInfo$SC)
est_cov <- estimateD(abund_list, q = c(0, 1, 2), datatype = "abundance", base = "coverage", level = cov_min)
print(est_cov); save_tab(est_cov, "T21_inext_equal_coverage")

## Diversidade beta e sua partição (Baselga 2010, 2013)
Partição da diversidade beta entre as parcelas de cada mix (Baselga 2010, 2013):
- Sørensen = **substituição** de táxons (turnover) + **aninhamento**.
- Bray-Curtis = **variação balanceada** + **gradiente de abundância**.

In [ ]:
%%R
# Entre as 8 parcelas de cada mix, média dos pares:
#   presença/ausência (Sørensen) = substituição (Simpson) + aninhamento;
#   abundância (Bray-Curtis) = variação balanceada (troca de indivíduos entre táxons) + gradiente de abundância.
beta_pairs <- function(m) {
  idx <- combn(nrow(m), 2)
  bind_rows(lapply(seq_len(ncol(idx)), function(k) {
    x <- m[idx[1, k], ]; y <- m[idx[2, k], ]
    a <- sum(x > 0 & y > 0); b <- sum(x > 0 & y == 0); c <- sum(x == 0 & y > 0)
    A <- sum(pmin(x, y)); B <- sum(x) - A; C <- sum(y) - A
    data.frame(sor = (b + c) / (2 * a + b + c), sim = min(b, c) / (a + min(b, c)),
               bc = (B + C) / (2 * A + B + C), bal = min(B, C) / (A + min(B, C)))
  })) %>% mutate(nes = sor - sim, gra = bc - bal)
}
beta_mix <- bind_rows(lapply(levels(env$mix), function(mx) {
  beta_pairs(mat[env$mix == mx, , drop = FALSE]) %>% mutate(mix = mx)
}))
beta_sum <- beta_mix %>% group_by(mix) %>%
  summarise(across(c(sor, sim, nes, bc, bal, gra), list(mean = mean, se = ~ sd(.) / sqrt(n()))))
print(as.data.frame(beta_sum)); save_tab(beta_sum, "T22_beta_partition_within_mix")
# Teste: a dissimilaridade entre parcelas do mesmo mix difere entre mixes? (betadisper em Bray-Curtis bruto)
print(permutest(betadisper(vegdist(mat, "bray"), env$mix), permutations = 9999))

## Rede mix × táxon (métricas da rede bipartida)
Rede bipartida mix × táxon com conectância, aninhamento (NODF), especialização H2' e d' por táxon (pacote bipartite).

In [ ]:
%%R
web <- t(sapply(split(as.data.frame(mat), env$mix), colSums))
print(web)
net <- tryCatch({
  library(bipartite)
  nl <- networklevel(web, index = c("connectance", "H2", "weighted NODF", "Shannon diversity"))
  sl <- specieslevel(web, index = c("d", "degree"), level = "higher")
  print(nl); print(sl)
  save_tab(data.frame(metric = names(nl), value = nl), "T23_network_metrics")
  save_tab(cbind(taxon = rownames(sl), sl), "T24_network_taxon_specialization")
  nl
}, error = function(e) { cat("bipartite não disponível:", conditionMessage(e), "\n"); NULL })

## H5 – Estabilidade temporal (CV e sincronia por parcela)
H5: coeficiente de variação da abundância nas 13 avaliações e sincronia entre táxons (Loreau & de Mazancourt 2008) por parcela.

In [ ]:
%%R
ts_plot <- dat %>% group_by(plot_id, block, mix, eval, taxon) %>% summarise(n = sum(count), .groups = "drop")
synchrony <- function(df) {
  w <- df %>% pivot_wider(names_from = taxon, values_from = n, values_fill = 0) %>% arrange(eval)
  m <- as.matrix(w[, -1]); m <- m[, colSums(m) > 0, drop = FALSE]; tot <- rowSums(m)
  c(cv = sd(tot) / mean(tot), synchrony = var(tot) / sum(apply(m, 2, sd))^2)
}
stab <- ts_plot %>% group_by(plot_id, block, mix) %>%
  group_modify(~ as.data.frame(t(synchrony(select(.x, eval, taxon, n))))) %>% ungroup()
stab_tests <- bind_rows(lapply(c("cv", "synchrony"), function(v) {
  a <- as.data.frame(Anova(lm(as.formula(paste(v, "~ mix + block")), data = stab), type = 2))
  a$term <- rownames(a); rownames(a) <- NULL; a$response <- v; a
}))
print(stab %>% group_by(mix) %>% summarise(across(c(cv, synchrony), mean)))
print(stab_tests %>% filter(term == "mix")); save_tab(stab, "T25_temporal_stability")

## Correlação entre diversidade e microclima por mix (Spearman)
- Índices calculados em cada observação (data × parcela): abundância, riqueza (q = 0), diversidade de Shannon (q = 1) e de Simpson (q = 2), equabilidade de Pielou e proporção de polinizadores.
- Correlação de Spearman com temperatura, umidade, vento e luminosidade, separada por mix e com todos os mixes juntos. Os p-valores são corrigidos por Holm dentro de cada mix.
- A **correlação parcial** desconta os dias após a semeadura, porque o clima e a floração mudam juntos ao longo da estação.

In [ ]:
%%R
# Índices por observação (data × parcela); observações sem nenhum visitante saem (índices indefinidos)
obs_div <- dat %>% group_by(date, das, block, plot_id, mix, temp, hum, wind, lux) %>%
  summarise(N = sum(count), q0 = sum(count > 0),
            q1 = { p <- count[count > 0] / sum(count); if (sum(count) > 0) exp(-sum(p * log(p))) else NA_real_ },
            q2 = { p <- count[count > 0] / sum(count); if (sum(count) > 0) 1 / sum(p^2) else NA_real_ },
            prop_poll = sum(count[fg == "Pollinators"]) / sum(count), .groups = "drop") %>%
  filter(N > 0) %>% mutate(pielou = ifelse(q0 > 1, log(q1) / log(q0), NA_real_))
idx_en  <- c(N = "Abundance", q0 = "Richness (q = 0)", q1 = "Shannon diversity (q = 1)",
             q2 = "Simpson diversity (q = 2)", pielou = "Pielou evenness", prop_poll = "Proportion of pollinators")
clim_en <- c(temp = "Air temperature", hum = "Relative humidity", wind = "Wind speed", lux = "Light intensity")

# Correlação de Spearman simples e parcial (descontando os dias após a semeadura, que mudam clima e floração juntos)
spear <- function(d) {
  bind_rows(lapply(names(idx_en), function(i) bind_rows(lapply(names(clim_en), function(cv) {
    ok <- complete.cases(d[[i]], d[[cv]]); x <- d[[i]][ok]; y <- d[[cv]][ok]; t <- d$das[ok]
    ct <- suppressWarnings(cor.test(x, y, method = "spearman", exact = FALSE))
    rx <- resid(lm(rank(x) ~ rank(t))); ry <- resid(lm(rank(y) ~ rank(t)))
    pc <- cor.test(rx, ry)
    data.frame(index = i, climate = cv, n = sum(ok), rho = unname(ct$estimate), p = ct$p.value,
               rho_partial = unname(pc$estimate), p_partial = pc$p.value)
  }))))
}
cor_mix <- bind_rows(
  obs_div %>% group_by(mix) %>% group_modify(~ spear(.x)) %>% ungroup() %>% mutate(mix = as.character(mix)),
  spear(obs_div) %>% mutate(mix = "All mixes")) %>%
  group_by(mix) %>% mutate(p_holm = p.adjust(p, "holm"), p_partial_holm = p.adjust(p_partial, "holm")) %>% ungroup()
print(cor_mix %>% mutate(across(where(is.numeric), ~ round(., 3))), n = 100)
save_tab(cor_mix, "T27_spearman_diversity_climate_by_mix")

# Correlação entre as próprias variáveis climáticas (justifica tirar a umidade dos modelos)
print(round(cor(obs %>% select(temp, hum, wind, lux), method = "spearman"), 2))

## Análise multivariada: RDA comunidade × microclima e partição da variância
- **RDA parcial** (transformação de Hellinger) da composição de cada observação explicada pelo mix e pelo microclima, descontando o bloco. O microclima é testado com permutações dentro da parcela.
- Teste de sensibilidade: o microclima depois de descontar a tendência sazonal.
- **Partição da variância** (Borcard et al. 1992; Peres-Neto et al. 2006) entre mix, microclima e tempo (dias após a semeadura).

In [ ]:
%%R
# Matriz por observação (data × parcela), transformação de Hellinger (Legendre & Gallagher 2001)
comm_obs <- dat %>% group_by(date, plot_id, taxon) %>% summarise(n = sum(count), .groups = "drop") %>%
  pivot_wider(names_from = taxon, values_from = n, values_fill = 0) %>%
  left_join(obs %>% select(date, plot_id, block, mix, das, temp, hum, wind, lux, temp_s, hum_s, wind_s, lux_s),
            by = c("date", "plot_id"))
Y_raw <- comm_obs %>% select(all_of(unique(dat$taxon))) %>% as.matrix()
keep  <- rowSums(Y_raw) > 0
Y <- decostand(Y_raw[keep, ], "hellinger"); E <- as.data.frame(comm_obs[keep, ]) %>% mutate(das2 = das^2)
cat("Observações com visitantes:", nrow(Y), "\n")

# RDA parcial: mix e microclima, descontando o bloco
rda_full <- rda(Y ~ mix + temp_s + wind_s + lux_s + Condition(block), data = E)
print(vif.cca(rda_full))
cat("R² ajustado da RDA:", round(RsquareAdj(rda_full)$adj.r.squared, 3), "\n")
# Teste do microclima com permutações dentro da parcela (as observações repetidas da mesma parcela não são independentes)
an_rda <- anova(rda_full, by = "margin", permutations = how(nperm = 999, blocks = E$plot_id))
an_rda <- cbind(term = rownames(an_rda), as.data.frame(an_rda)) %>% filter(term != "mix")
print(an_rda); save_tab(an_rda, "T28_rda_climate_margin")
# O efeito do mix na composição é testado no nível da parcela (PERMANOVA, T12), pois o mix não varia dentro da parcela.

# Sensibilidade: o microclima ainda explica a composição depois de descontar a tendência sazonal?
rda_season <- rda(Y ~ temp_s + wind_s + lux_s + Condition(block + mix + das + das2), data = E)
an_season <- anova(rda_season, by = "margin", permutations = how(nperm = 999, blocks = E$plot_id))
an_season <- cbind(term = rownames(an_season), as.data.frame(an_season)); print(an_season)
save_tab(an_season, "T29_rda_climate_after_season")

# Partição da variância (Borcard et al. 1992; Peres-Neto et al. 2006): mix × microclima × tempo
vp <- varpart(Y, ~ mix, ~ temp_s + wind_s + lux_s, ~ das + das2, data = E)
print(vp)
vp_tab <- data.frame(fraction = rownames(vp$part$indfract), adj_R2 = vp$part$indfract$Adj.R.square)
vp_tab$fraction <- c("Mix only", "Microclimate only", "Time only", "Mix + microclimate", "Microclimate + time",
                     "Mix + time", "All three", "Residuals")
print(vp_tab); save_tab(vp_tab, "T30_variance_partitioning")

# Escores para a figura (escala simétrica)
rda_plot <- rda(Y ~ temp_s + wind_s + lux_s + mix, data = E)
sc_obs <- cbind(as.data.frame(scores(rda_plot, display = "sites", scaling = "symmetric")), mix = E$mix)
sc_tx  <- as.data.frame(scores(rda_plot, display = "species", scaling = "symmetric")); sc_tx$taxon <- rownames(sc_tx)
sc_env <- as.data.frame(scores(rda_plot, display = "bp", scaling = "symmetric")); sc_env$var <- rownames(sc_env)
ax_var <- round(100 * rda_plot$CCA$eig[1:2] / rda_plot$tot.chi, 1)

## Espécies de plantas de cada mix (proporção na semeadura)
- Espécies e proporção de sementes de cada mix (organização do plantio de 12/07/2025 e Tabela 2 do projeto).
- Conta quantas espécies têm flores visitadas por insetos (todas menos as gramíneas, polinizadas pelo vento).
- **Confira:** a crotalária dos mixes Carbono e Polinizador está como *Crotalaria* sp. e a composição do Multipropósito segue a Tabela 2 do projeto.

In [ ]:
%%R
# Composição das sementes (organização do plantio de 12/07/2025 e Tabela 2 do projeto de tese).
# Carbono e Polinizador: % do peso de sementes; Multipropósito: mix comercial com 11 espécies.
plants <- tribble(
  ~mix,           ~common,               ~species,                     ~family,         ~pct,
  "Carbon",       "Crotalária",          "Crotalaria sp.",             "Fabaceae",      20,
  "Carbon",       "Feijão-caupi",        "Vigna unguiculata",          "Fabaceae",      20,
  "Carbon",       "Milheto",             "Pennisetum glaucum",         "Poaceae",       15,
  "Carbon",       "Capim Marandu",       "Urochloa brizantha",         "Poaceae",       15,
  "Carbon",       "Capim Llanero",       "Urochloa dictyoneura",       "Poaceae",       15,
  "Carbon",       "Capim Zuri",          "Megathyrsus maximus",        "Poaceae",       15,
  "Pollinator",   "Girassol",            "Helianthus annuus",          "Asteraceae",    20,
  "Pollinator",   "Feijão-caupi",        "Vigna unguiculata",          "Fabaceae",      20,
  "Pollinator",   "Crotalária",          "Crotalaria sp.",             "Fabaceae",      20,
  "Pollinator",   "Sorgo",               "Sorghum bicolor",            "Poaceae",       10,
  "Pollinator",   "Milheto",             "Pennisetum glaucum",         "Poaceae",       15,
  "Pollinator",   "Capim Zuri",          "Megathyrsus maximus",        "Poaceae",       15,
  "Multipurpose", "Crotalária juncea",   "Crotalaria juncea",          "Fabaceae",      10,
  "Multipurpose", "Crotalária ochroleuca","Crotalaria ochroleuca",     "Fabaceae",      10,
  "Multipurpose", "Crotalária spectabilis","Crotalaria spectabilis",   "Fabaceae",      10,
  "Multipurpose", "Feijão-guandu",       "Cajanus cajan",              "Fabaceae",      10,
  "Multipurpose", "Feijão-caupi",        "Vigna unguiculata",          "Fabaceae",       5,
  "Multipurpose", "Crambe",              "Crambe abyssinica",          "Brassicaceae",  10,
  "Multipurpose", "Nabo forrageiro",     "Raphanus sativus",           "Brassicaceae",  10,
  "Multipurpose", "Braquiária ruziziensis","Urochloa ruziziensis",     "Poaceae",       10,
  "Multipurpose", "Milheto",             "Pennisetum glaucum",         "Poaceae",       10,
  "Multipurpose", "Girassol",            "Helianthus annuus",          "Asteraceae",    10,
  "Multipurpose", "Trigo mourisco",      "Fagopyrum esculentum",       "Polygonaceae",   5) %>%
  mutate(mix = factor(mix, levels = c("Carbon", "Multipurpose", "Pollinator")),
         # Poaceae é polinizada pelo vento; as demais famílias oferecem néctar e/ou pólen a insetos
         flower_resource = ifelse(family == "Poaceae", "Wind-pollinated (grasses)", "Insect-visited flowers"))
stopifnot(all(tapply(plants$pct, plants$mix, sum) == 100))
plant_sum <- plants %>% group_by(mix) %>%
  summarise(n_species = n(), n_families = n_distinct(family),
            n_insect_flowers = sum(flower_resource == "Insect-visited flowers"),
            pct_insect_flowers = sum(pct[flower_resource == "Insect-visited flowers"]), .groups = "drop")
print(plant_sum)
save_tab(plants, "T31_cover_crop_species_by_mix"); save_tab(plant_sum, "T32_cover_crop_mix_summary")

## Causa e efeito: clima → abundância → diversidade (modelos mistos e análise de caminhos)
- **Modelos mistos** de cada índice de diversidade por observação com temperatura, vento e luminosidade padronizados, descontando mix, bloco, parcela e tendência sazonal. A umidade entra num modelo alternativo no lugar da temperatura (r = −0,96).
- **Análise de caminhos** (SEM por partes; Lefcheck 2016; Shipley 2009): dias após a semeadura → microclima → abundância → riqueza e diversidade de Shannon. Separa o efeito **direto** do clima na diversidade do efeito **indireto** via abundância.
- O teste C de Fisher verifica se os caminhos ausentes (mix → microclima) fazem falta; p > 0,05 indica bom ajuste.

In [ ]:
%%R
# Base: observações com visitantes (obs_div, da célula de correlação). Todas as variáveis padronizadas (z),
# então os coeficientes são comparáveis: variação em desvios-padrão da resposta por desvio-padrão do clima.
z <- function(x) as.numeric(scale(x))
sem_d <- obs_div %>%
  mutate(logN = log(N), das_z = z(das), das2_z = z(das_z^2),
         temp_z = z(temp), wind_z = z(wind), lux_z = z(lux), hum_z = z(hum),
         across(c(logN, q0, q1, q2, pielou, prop_poll), z, .names = "{.col}_z"))

# (1) Efeito líquido do clima em cada índice, descontando mix, bloco, parcela e tendência sazonal
resp_en <- c(logN_z = "Abundance (log)", q0_z = "Richness (q = 0)", q1_z = "Shannon diversity (q = 1)",
             q2_z = "Simpson diversity (q = 2)", pielou_z = "Pielou evenness", prop_poll_z = "Proportion of pollinators")
fit_lmm <- function(f, d) glmmTMB(as.formula(f), data = d)
clim_eff <- bind_rows(lapply(names(resp_en), function(r) {
  d <- sem_d %>% filter(!is.na(.data[[r]]))
  m <- fit_lmm(paste(r, "~ temp_z + wind_z + lux_z + mix + das_z + das2_z + block + (1 | plot_id)"), d)
  m_h <- fit_lmm(paste(r, "~ hum_z + wind_z + lux_z + mix + das_z + das2_z + block + (1 | plot_id)"), d)
  co <- summary(m)$coefficients$cond; ch <- summary(m_h)$coefficients$cond
  r2 <- performance::r2(m); m0 <- fit_lmm(paste(r, "~ mix + das_z + das2_z + block + (1 | plot_id)"), d)
  lrt <- anova(m0, m)
  bind_rows(
    data.frame(response = r, term = c("temp_z", "wind_z", "lux_z"), beta = co[c("temp_z", "wind_z", "lux_z"), 1],
               se = co[c("temp_z", "wind_z", "lux_z"), 2], p = co[c("temp_z", "wind_z", "lux_z"), 4]),
    data.frame(response = r, term = "hum_z", beta = ch["hum_z", 1], se = ch["hum_z", 2], p = ch["hum_z", 4])) %>%
    mutate(n = nrow(d), R2_marginal = r2$R2_marginal, R2_conditional = r2$R2_conditional,
           LRT_climate_chisq = lrt$Chisq[2], LRT_climate_p = lrt$`Pr(>Chisq)`[2])
})) %>% mutate(low = beta - 1.96 * se, high = beta + 1.96 * se)
rownames(clim_eff) <- NULL
print(clim_eff %>% mutate(across(where(is.numeric), ~ round(., 3))))
save_tab(clim_eff, "T33_climate_effects_on_diversity")

# (2) Análise de caminhos (SEM por partes; Lefcheck 2016): estação → microclima → abundância → diversidade.
# Cada equação é um modelo misto com a parcela como efeito aleatório; mix e bloco entram como covariáveis.
eq <- list(
  temp_z = "temp_z ~ das_z + das2_z + block + (1 | plot_id)",
  wind_z = "wind_z ~ das_z + das2_z + block + (1 | plot_id)",
  lux_z  = "lux_z ~ das_z + das2_z + (1 | plot_id)",
  logN_z = "logN_z ~ temp_z + wind_z + lux_z + das_z + das2_z + mix + block + (1 | plot_id)",
  q0_z   = "q0_z ~ logN_z + temp_z + wind_z + lux_z + das_z + das2_z + mix + block + (1 | plot_id)",
  q1_z   = "q1_z ~ logN_z + temp_z + wind_z + lux_z + das_z + das2_z + mix + block + (1 | plot_id)")
sem_m <- lapply(eq, fit_lmm, d = sem_d)
paths <- bind_rows(lapply(names(sem_m), function(r) {
  co <- summary(sem_m[[r]])$coefficients$cond
  co <- co[!grepl("Intercept|^mix|^block", rownames(co)), , drop = FALSE]
  data.frame(to = r, from = rownames(co), beta = co[, 1], se = co[, 2], p = co[, 4])
}))
rownames(paths) <- NULL

# Teste de ajuste (separação direcionada, Shipley 2009): caminhos ausentes no modelo.
# O bloco entra nas equações do microclima porque as medições seguem a ordem dos blocos ao longo do dia.
# Afirmações independentes: o mix não altera o microclima, dado o dia da estação e o bloco.
dsep <- bind_rows(lapply(c("temp_z", "wind_z", "lux_z"), function(cv) {
  m <- fit_lmm(paste(cv, "~ das_z + das2_z + block + mix + (1 | plot_id)"), sem_d)
  a <- as.data.frame(Anova(m, type = 2))
  data.frame(claim = paste("mix ->", cv), p = a["mix", "Pr(>Chisq)"])
}))
fisher_C <- -2 * sum(log(dsep$p)); df_C <- 2 * nrow(dsep)
sem_fit <- data.frame(Fisher_C = fisher_C, df = df_C, p = pchisq(fisher_C, df_C, lower.tail = FALSE))
print(dsep); print(sem_fit)
cat("p > 0,05 no teste C de Fisher = o modelo de caminhos é consistente com os dados.\n")

# Efeitos indiretos do clima sobre a diversidade via abundância (produto dos coeficientes; erro pelo método delta)
ind <- bind_rows(lapply(c("q0_z", "q1_z"), function(r) bind_rows(lapply(c("temp_z", "wind_z", "lux_z"), function(cv) {
  a <- paths %>% filter(to == "logN_z", from == cv); b <- paths %>% filter(to == r, from == "logN_z")
  d <- paths %>% filter(to == r, from == cv)
  est <- a$beta * b$beta; se <- sqrt(a$beta^2 * b$se^2 + b$beta^2 * a$se^2)
  data.frame(response = r, climate = cv, direct = d$beta, p_direct = d$p, indirect = est,
             p_indirect = 2 * pnorm(-abs(est / se)), total = d$beta + est)
}))))
print(ind %>% mutate(across(where(is.numeric), ~ round(., 3))))
r2_sem <- sapply(sem_m, function(m) as.numeric(performance::r2(m)$R2_marginal))
print(round(r2_sem, 3))
save_tab(paths %>% mutate(R2_marginal_response = r2_sem[to]), "T34_path_coefficients")
save_tab(bind_rows(dsep, data.frame(claim = "Fisher's C", p = sem_fit$p)) %>%
           mutate(C = c(rep(NA, nrow(dsep)), fisher_C), df = c(rep(NA, nrow(dsep)), df_C)), "T35_path_model_fit")
save_tab(ind, "T36_direct_indirect_climate_effects")

## Biomassa das plantas de cobertura × diversidade dos polinizadores
- Lê a planilha de biomassa (M1 = Carbono, M2 = Polinizador, M3 = Multipropósito) e confere a correspondência com as parcelas dos polinizadores. As parcelas S+F (sem mix) ficam de fora.
- Compara a biomassa entre mixes (com o sistema soja × rotação como covariável).
- Testa o efeito da biomassa na abundância, riqueza, diversidade e composição dos visitantes, sozinha e depois de descontar o mix.

In [ ]:
%%R
# Planilha de biomassa: M1 = Carbono, M2 = Polinizador, M3 = Multipropósito; parcelas S+F (soja em pousio) ficam de fora.
bio <- read.csv(ARQUIVO_BIOMASSA) %>%
  filter(Mix %in% c("M1", "M2", "M3")) %>%
  transmute(block = factor(Block), plot = Plot,
            mix_bio = factor(recode(Mix, M1 = "Carbon", M2 = "Pollinator", M3 = "Multipurpose"),
                             levels = c("Carbon", "Multipurpose", "Pollinator")),
            system = factor(System), biomass = Fresh_mass_kg_ha / 1000)   # Mg ha-1 de massa fresca
plot_key <- obs %>% distinct(block, plot, plot_id, mix)
bio <- bio %>% left_join(plot_key, by = c("block", "plot"))
stopifnot(nrow(bio) == 24, !anyNA(bio$plot_id), all(bio$mix == bio$mix_bio))   # confere a correspondência das parcelas
bio <- bio %>% select(-mix_bio) %>% mutate(biomass_s = as.numeric(scale(biomass)))
cat("Biomassa (Mg/ha) por mix:\n"); print(bio %>% group_by(mix) %>% summarise(mean = mean(biomass), se = sd(biomass) / sqrt(n())))

# (1) Os mixes diferem em biomassa? (o sistema soja × rotação entra como covariável)
m_bio <- lm(biomass ~ mix + system + block, data = bio)
an_bio <- as.data.frame(Anova(m_bio, type = 2)); an_bio$term <- rownames(an_bio); print(an_bio)
emm_bio <- letras(emmeans(m_bio, ~ mix)); print(emm_bio)
save_tab(an_bio, "T37_anova_biomass"); save_tab(emm_bio, "T38_emmeans_biomass")

# (2) Diversidade por parcela (soma das 12 avaliações) em função da biomassa
plot_div <- div %>% select(plot_id, N, q0, q1, q2, pielou) %>%
  left_join(obs_fg %>% group_by(plot_id) %>% summarise(prop_poll = sum(abund[fg == "Pollinators"]) / sum(abund)), by = "plot_id") %>%
  left_join(bio, by = "plot_id")
idx_plot <- c(N = "Total visits", q0 = "Richness (q = 0)", q1 = "Shannon diversity (q = 1)",
              q2 = "Simpson diversity (q = 2)", pielou = "Pielou evenness", prop_poll = "Proportion of pollinators")
bio_tests <- bind_rows(lapply(names(idx_plot), function(v) {
  m1 <- lm(as.formula(paste(v, "~ biomass + block")), data = plot_div)          # só a biomassa
  m2 <- lm(as.formula(paste(v, "~ mix + biomass + system + block")), data = plot_div)   # biomassa além do mix
  a1 <- Anova(m1, type = 2); a2 <- Anova(m2, type = 2)
  ct <- suppressWarnings(cor.test(plot_div[[v]], plot_div$biomass, method = "spearman", exact = FALSE))
  data.frame(index = v, slope_per_Mg = coef(m1)["biomass"], p_biomass = a1["biomass", "Pr(>F)"],
             slope_adj_mix = coef(m2)["biomass"], p_biomass_adj_mix = a2["biomass", "Pr(>F)"],
             p_mix_adj_biomass = a2["mix", "Pr(>F)"], p_system = a2["system", "Pr(>F)"],
             rho = unname(ct$estimate), p_rho = ct$p.value)
}))
rownames(bio_tests) <- NULL
print(bio_tests %>% mutate(across(where(is.numeric), ~ round(., 3))))
save_tab(bio_tests, "T39_biomass_diversity_plot_level")

# (3) Visitas por observação: a biomassa entra no modelo principal de abundância
obs_bio <- obs %>% left_join(bio %>% select(plot_id, biomass, biomass_s, system), by = "plot_id")
m_tot_bio  <- fit_nb("abund ~ mix + biomass_s + system + temp_s + wind_s + lux_s + poly(day_s, 2) + block", obs_bio)
m_tot_bio2 <- fit_nb("abund ~ biomass_s + system + temp_s + wind_s + lux_s + poly(day_s, 2) + block", obs_bio)
an_bio_obs <- bind_rows(
  as.data.frame(Anova(m_tot_bio, type = 2)) %>% mutate(term = rownames(.), model = "Mix + biomass"),
  as.data.frame(Anova(m_tot_bio2, type = 2)) %>% mutate(term = rownames(.), model = "Biomass without mix"))
rownames(an_bio_obs) <- NULL; print(an_bio_obs)
irr_bio <- exp(fixef(m_tot_bio2)$cond["biomass_s"]); cat("IRR por 1 DP de biomassa (sem mix):", round(irr_bio, 3), "\n")
save_tab(an_bio_obs, "T40_abundance_model_with_biomass")

# (4) Composição: a biomassa explica a composição dos visitantes? (PERMANOVA, permutações dentro do bloco)
env_b <- env %>% left_join(bio %>% select(plot_id, biomass, system), by = "plot_id")
pmv_bio <- adonis2(d_bc ~ mix + biomass + system, data = env_b, by = "margin",
                   permutations = how(nperm = 9999, blocks = env_b$block))
print(pmv_bio); save_tab(cbind(term = rownames(pmv_bio), as.data.frame(pmv_bio)), "T41_permanova_biomass")

## Ícones dos visitantes florais (desenhos vetoriais; podem ser trocados por fotos)
- Desenhos simples de cada visitante (vista de cima) e de cada planta de cobertura. Os visitantes aparecem na figura por táxon e na rede; as plantas, na figura das espécies dos mixes.
- **Para usar fotos ou ilustrações próprias:** salve um PNG (de preferência com fundo transparente) na pasta **04_Figuras/Icones_polinizadores** do Drive, com o nome que esta célula mostra (ex.: `Apis_mellifera.png` ou `Helianthus_annuus.png`). Na próxima execução o notebook usa a imagem no lugar do desenho.

In [ ]:
%%R
# Cada táxon recebe um desenho simples (vista de cima). Para usar uma foto ou ilustração própria, coloque um PNG
# com o nome indicado em icon_file (ex.: "Apis_mellifera.png") na pasta 04_Figuras/Icones_polinizadores do Drive:
# o notebook baixa a pasta e usa a imagem no lugar do desenho.
library(grid)
ell <- function(cx, cy, rx, ry, ang = 0, n = 80) {
  t <- seq(0, 2 * pi, length.out = n); a <- ang * pi / 180
  data.frame(x = cx + rx * cos(t) * cos(a) - ry * sin(t) * sin(a), y = cy + rx * cos(t) * sin(a) + ry * sin(t) * cos(a))
}
pg <- function(d, fill, col = "grey15", lwd = 0.6, alpha = 1)
  polygonGrob(d$x, d$y, gp = gpar(fill = adjustcolor(fill, alpha), col = col, lwd = lwd))
# Faixas do abdômen: parte da elipse entre y1 e y2
band <- function(cx, cy, rx, ry, y1, y2, n = 30) {
  y <- seq(max(y1, cy - ry), min(y2, cy + ry), length.out = n)
  w <- rx * sqrt(pmax(0, 1 - ((y - cy) / ry)^2))
  data.frame(x = c(cx - w, rev(cx + w)), y = c(y, rev(y)))
}
legs <- function(cx, cy, sp = 0.17, col = "grey15") {
  g <- list()
  for (s in c(-1, 1)) for (k in 1:3) {
    dy <- c(0.05, 0, -0.05)[k]; ex <- c(0.13, 0.17, 0.15)[k]; ey <- c(0.1, -0.02, -0.14)[k]
    g[[length(g) + 1]] <- linesGrob(c(cx, cx + s * sp * 0.6, cx + s * ex), c(cy + dy, cy + dy + ey * 0.4, cy + dy + ey),
                                    gp = gpar(col = col, lwd = 0.7))
  }
  g
}
antennae <- function(cx, cy, col = "grey15", len = 0.1)
  list(linesGrob(c(cx - 0.02, cx - 0.05, cx - 0.08), c(cy, cy + len * 0.6, cy + len), gp = gpar(col = col, lwd = 0.7)),
       linesGrob(c(cx + 0.02, cx + 0.05, cx + 0.08), c(cy, cy + len * 0.6, cy + len), gp = gpar(col = col, lwd = 0.7)))

# Abelha genérica: cor do corpo, das faixas e das asas; escala (abelhas pequenas × grandes)
bee <- function(body, thorax = body, stripe = NA, wing = "#CFE3F2", sc = 1, eye = "grey10") {
  ab <- c(0.5, 0.36, 0.14, 0.21)
  g <- c(legs(0.5, 0.6),
         list(pg(ell(0.33, 0.55, 0.2, 0.08, -25), wing, "grey40", 0.4, 0.75),
              pg(ell(0.67, 0.55, 0.2, 0.08, 25), wing, "grey40", 0.4, 0.75),
              pg(ell(ab[1], ab[2], ab[3], ab[4]), body)))
  if (!is.na(stripe)) for (yy in c(0.22, 0.31, 0.40))
    g <- c(g, list(pg(band(ab[1], ab[2], ab[3], ab[4], yy, yy + 0.045), stripe, NA)))
  g <- c(g, list(pg(ell(0.5, 0.62, 0.11, 0.1), thorax), pg(ell(0.5, 0.78, 0.075, 0.065), body)),
         antennae(0.5, 0.83),
         list(pg(ell(0.46, 0.79, 0.022, 0.035), eye, NA), pg(ell(0.54, 0.79, 0.022, 0.035), eye, NA)))
  gTree(children = do.call(gList, g), vp = viewport(width = sc, height = sc))
}
hoverfly <- function() {
  ab <- c(0.5, 0.36, 0.13, 0.2)
  g <- c(legs(0.5, 0.6),
         list(pg(ell(0.36, 0.5, 0.21, 0.06, -15), "#DCE9F5", "grey40", 0.4, 0.75),
              pg(ell(0.64, 0.5, 0.21, 0.06, 15), "#DCE9F5", "grey40", 0.4, 0.75),
              pg(ell(ab[1], ab[2], ab[3], ab[4]), "grey12")))
  for (yy in c(0.25, 0.34, 0.43)) g <- c(g, list(pg(band(ab[1], ab[2], ab[3], ab[4], yy, yy + 0.04), "#F2C230", NA)))
  g <- c(g, list(pg(ell(0.5, 0.62, 0.1, 0.09), "#5A4A2A"), pg(ell(0.5, 0.77, 0.085, 0.06), "#8B3A1E"),
                 pg(ell(0.45, 0.78, 0.04, 0.05), "#A0301C", NA), pg(ell(0.55, 0.78, 0.04, 0.05), "#A0301C", NA)))
  do.call(gList, g)
}
beetle <- function(col = "#3B2F2A") {
  do.call(gList, c(legs(0.5, 0.55, col = "grey15"), antennae(0.5, 0.8),
                   list(pg(ell(0.5, 0.42, 0.17, 0.24), col), linesGrob(c(0.5, 0.5), c(0.19, 0.64), gp = gpar(col = "grey60", lwd = 0.6)),
                        pg(ell(0.5, 0.68, 0.11, 0.07), col), pg(ell(0.5, 0.78, 0.06, 0.045), col))))
}
butterfly <- function(col = "#E07B24") {
  do.call(gList, c(antennae(0.5, 0.72, len = 0.15),
                   list(pg(ell(0.32, 0.6, 0.18, 0.14, 25), col), pg(ell(0.68, 0.6, 0.18, 0.14, -25), col),
                        pg(ell(0.36, 0.35, 0.13, 0.1, -30), col), pg(ell(0.64, 0.35, 0.13, 0.1, 30), col),
                        pg(ell(0.27, 0.66, 0.04, 0.035), "grey10", NA), pg(ell(0.73, 0.66, 0.04, 0.035), "grey10", NA),
                        pg(ell(0.5, 0.48, 0.03, 0.24), "grey15"))))
}
wasp <- function(col = "grey12", wing = "#B9C3CC") {
  do.call(gList, c(legs(0.5, 0.62), antennae(0.5, 0.86),
                   list(pg(ell(0.36, 0.55, 0.18, 0.05, -20), wing, "grey40", 0.4, 0.7),
                        pg(ell(0.64, 0.55, 0.18, 0.05, 20), wing, "grey40", 0.4, 0.7),
                        pg(ell(0.5, 0.3, 0.08, 0.17), col), pg(ell(0.5, 0.49, 0.025, 0.04), col),
                        pg(ell(0.5, 0.63, 0.075, 0.08), col), pg(ell(0.5, 0.78, 0.065, 0.055), col))))
}
icon_draw <- list(
  "Apoidea morphotype 1"          = function() bee("grey15", thorax = "grey25", sc = 0.75),
  "Apoidea morphotype 2"          = function() bee("grey10", thorax = "#4A3B2C", sc = 0.95),
  "Apis mellifera"                = function() bee("#C98A2E", thorax = "#7A5A35", stripe = "#4A3216"),
  "Xylocopa sp."                  = function() bee("grey8", thorax = "grey15", wing = "#7E7A9A", sc = 1.1),
  "Trigona pallens"               = function() bee("#C9A267", thorax = "#A87F45", wing = "#E5EEF5", sc = 0.65),
  "Halictidae (green morphotype)" = function() bee("#2E8B57", thorax = "#3BAA6A", stripe = "#1B5E3A", sc = 0.75),
  "Bombus sp."                    = function() bee("#F2C230", thorax = "grey15", stripe = "grey15"),
  "Syrphidae"                     = function() hoverfly(),
  "Coleoptera"                    = function() beetle(),
  "Lepidoptera"                   = function() butterfly(),
  "Hymenoptera morphotype 1"      = function() wasp(),
  "Hymenoptera morphotype 2"      = function() wasp("#5B3A29"),
  "Hymenoptera morphotype 3"      = function() wasp("#8A6A2F"))
icon_file <- function(tx) paste0(gsub("[^A-Za-z0-9]+", "_", sub(" sp\\.$", "", tx)), ".png")
icon_grob <- function(tx, dir = "icons") {
  f <- file.path(dir, icon_file(tx))
  if (file.exists(f)) {
    img <- png::readPNG(f)
    return(grobTree(rasterGrob(img, interpolate = TRUE), vp = viewport(width = unit(1, "snpc"), height = unit(1, "snpc"))))
  }
  if (!tx %in% names(icon_draw)) return(nullGrob())
  gTree(children = gList(icon_draw[[tx]]()), vp = viewport(width = unit(1, "snpc"), height = unit(1, "snpc")))
}

# ---- Ícones das plantas de cobertura (desenhos simples; também podem ser trocados por PNG na mesma pasta)
stem <- function(x0 = 0.5, y0 = 0.05, x1 = 0.5, y1 = 0.6, col = "#3E7B27", lwd = 1.2)
  linesGrob(c(x0, x1), c(y0, y1), gp = gpar(col = col, lwd = lwd))
leaf <- function(cx, cy, rx, ry, ang, col = "#5E9E3A") pg(ell(cx, cy, rx, ry, ang), col, "#2F5E1E", 0.4)
pea <- function(petal = "#F2C230", keel = "#E3A21A", leaves = TRUE) {   # flor de leguminosa (estandarte, asas, quilha)
  g <- list(stem(y1 = 0.62))
  if (leaves) g <- c(g, list(leaf(0.33, 0.3, 0.13, 0.05, 30), leaf(0.67, 0.36, 0.13, 0.05, -30), leaf(0.36, 0.45, 0.1, 0.04, 40)))
  g <- c(g, list(pg(ell(0.5, 0.8, 0.17, 0.15), petal, "grey30", 0.4),
                 pg(ell(0.38, 0.66, 0.08, 0.1, 30), petal, "grey30", 0.4), pg(ell(0.62, 0.66, 0.08, 0.1, -30), petal, "grey30", 0.4),
                 pg(ell(0.5, 0.62, 0.05, 0.09), keel, "grey30", 0.4)))
  do.call(gList, g)
}
sunflower <- function() {
  g <- list(stem(y1 = 0.55, lwd = 1.6), leaf(0.33, 0.3, 0.14, 0.07, 25), leaf(0.67, 0.38, 0.14, 0.07, -25))
  for (a in seq(0, 330, 30)) {
    r <- a * pi / 180
    g <- c(g, list(pg(ell(0.5 + 0.17 * cos(r), 0.72 + 0.17 * sin(r), 0.09, 0.035, a), "#F6C21A", "#B98A0A", 0.3)))
  }
  do.call(gList, c(g, list(pg(ell(0.5, 0.72, 0.1, 0.1), "#6B3E1E", "grey20", 0.4))))
}
crucifer <- function(petal = "white") {   # flores de 4 pétalas em cacho (crambe, nabo)
  g <- list(stem(y1 = 0.7), leaf(0.32, 0.25, 0.14, 0.06, 20), leaf(0.68, 0.3, 0.14, 0.06, -20))
  for (p in list(c(0.5, 0.8, 0.09), c(0.32, 0.64, 0.07), c(0.68, 0.66, 0.07))) {
    for (a in c(45, 135, 225, 315)) {
      r <- a * pi / 180
      g <- c(g, list(pg(ell(p[1] + p[3] * 0.6 * cos(r), p[2] + p[3] * 0.6 * sin(r), p[3] * 0.55, p[3] * 0.4, a), petal, "grey45", 0.3)))
    }
    g <- c(g, list(pg(ell(p[1], p[2], p[3] * 0.2, p[3] * 0.2), "#E8C33A", NA)))
  }
  do.call(gList, g)
}
buckwheat <- function() {   # inflorescência de flores pequenas branco-rosadas
  g <- list(stem(y1 = 0.62), leaf(0.33, 0.32, 0.12, 0.08, 20), leaf(0.67, 0.4, 0.12, 0.08, -20))
  set.seed(3)
  for (i in 1:14) g <- c(g, list(pg(ell(0.5 + rnorm(1, 0, 0.09), 0.76 + rnorm(1, 0, 0.07), 0.045, 0.045), "#FBEFF2", "#C98FA0", 0.3)))
  do.call(gList, g)
}
grass <- function(head = "spike", col = "#C9B26B") {   # gramínea: folhas finas e inflorescência
  g <- list(leaf(0.4, 0.3, 0.3, 0.025, 70, "#6FA444"), leaf(0.6, 0.3, 0.3, 0.025, 110, "#6FA444"),
            leaf(0.35, 0.22, 0.22, 0.02, 45, "#6FA444"), leaf(0.65, 0.22, 0.22, 0.02, 135, "#6FA444"),
            stem(y1 = 0.62, col = "#6FA444"))
  if (head == "spike") g <- c(g, list(pg(ell(0.5, 0.78, 0.045, 0.17), col, "#7D6A2E", 0.4)))          # milheto
  if (head == "panicle") for (k in 0:5) g <- c(g, list(pg(ell(0.5 + c(-0.07, 0.07)[k %% 2 + 1], 0.66 + k * 0.05, 0.06, 0.03,
                                                               c(20, -20)[k %% 2 + 1]), col, "#6A4A20", 0.3)))  # sorgo
  if (head == "raceme") for (k in 0:2) g <- c(g, list(linesGrob(c(0.5, 0.5 + c(-0.15, 0.15, -0.12)[k + 1]),
                                                                c(0.6 + k * 0.1, 0.72 + k * 0.1), gp = gpar(col = "#8C7A3A", lwd = 1.2))))  # braquiárias
  do.call(gList, g)
}
plant_draw <- list(
  "Crotalaria sp."         = function() pea("#F2C230"),
  "Crotalaria juncea"      = function() pea("#F2C230"),
  "Crotalaria ochroleuca"  = function() pea("#F3E6A0", "#E2D07A"),
  "Crotalaria spectabilis" = function() pea("#F2C230", "#D98A14"),
  "Vigna unguiculata"      = function() pea("#D9C8E8", "#B9A3D0"),
  "Cajanus cajan"          = function() pea("#F2B630", "#C0392B"),
  "Helianthus annuus"      = function() sunflower(),
  "Crambe abyssinica"      = function() crucifer("white"),
  "Raphanus sativus"       = function() crucifer("#EBD9F0"),
  "Fagopyrum esculentum"   = function() buckwheat(),
  "Pennisetum glaucum"     = function() grass("spike"),
  "Sorghum bicolor"        = function() grass("panicle", "#B5652A"),
  "Urochloa brizantha"     = function() grass("raceme"),
  "Urochloa dictyoneura"   = function() grass("raceme"),
  "Urochloa ruziziensis"   = function() grass("raceme"),
  "Megathyrsus maximus"    = function() grass("panicle", "#B9A86A"))
icon_draw <- c(icon_draw, plant_draw)

# Geom para desenhar um ícone por linha de dados (funciona com facetas): aes(x, y, icon = nome);
# size em fração do painel (units = "snpc") ou em mm (units = "mm")
GeomIcon <- ggproto("GeomIcon", Geom, required_aes = c("x", "y", "icon"),
  default_aes = aes(size = 0.2, hjust = 0.5, vjust = 0.5),
  draw_key = draw_key_blank,
  draw_panel = function(data, panel_params, coord, units = "snpc") {
    d <- coord$transform(data, panel_params)
    gl <- lapply(seq_len(nrow(d)), function(i) {
      g <- icon_grob(d$icon[i])
      gTree(children = gList(g), vp = viewport(x = d$x[i], y = d$y[i], width = unit(d$size[i], units),
                                               height = unit(d$size[i], units), just = c(d$hjust[i], d$vjust[i])))
    })
    gTree(children = do.call(gList, gl))
  })
geom_icon <- function(mapping = NULL, data = NULL, ..., inherit.aes = FALSE)
  layer(geom = GeomIcon, mapping = mapping, data = data, stat = "identity", position = "identity",
        show.legend = FALSE, inherit.aes = inherit.aes, params = list(...))
cat("Ícones próprios (visitantes e plantas) encontrados:", if (dir.exists("icons")) list.files("icons") else "nenhum", "\n")
cat("Para trocar um desenho por foto, use estes nomes de arquivo:\n"); print(sapply(names(icon_draw), icon_file))

## Figuras finais (inglês, padrão AGEE)
Cada figura é salva em `out/figs` e enviada para 04_Figuras na última célula.

### Fig. – Visitas por táxon e mix (estilo Mallinger et al. 2019)

In [ ]:
%%R -w 190 -h 130 -u mm -r 150
bar_tx <- obs_tx %>% group_by(taxon, fg, mix, plot_id) %>% summarise(v = mean(count), .groups = "drop") %>%
  group_by(taxon, fg, mix) %>% summarise(mean = mean(v), se = sd(v) / sqrt(n()), .groups = "drop") %>%
  left_join(tx_emm %>% select(taxon, mix, .group), by = c("taxon", "mix")) %>%
  filter(taxon %in% main_taxa) %>%
  mutate(taxon_lab = factor(lab_taxon_chr(taxon), levels = lab_taxon_chr(main_taxa)))
fig_taxa <- ggplot(bar_tx, aes(mix, mean, fill = mix)) +
  geom_col(width = 0.7, colour = "black", linewidth = 0.2) +
  geom_errorbar(aes(ymin = mean - se, ymax = mean + se), width = 0.2, linewidth = 0.3) +
  geom_text(aes(y = mean + se, label = .group), vjust = -0.5, size = 2.4) +
  # ícone do táxon no canto superior esquerdo de cada painel (Inf/-Inf = borda do painel)
  geom_icon(data = distinct(bar_tx, taxon, taxon_lab), aes(x = -Inf, y = Inf, icon = taxon),
            size = 10, units = "mm", hjust = 0, vjust = 1) +
  facet_wrap(~ taxon_lab, scales = "free_y", labeller = label_parsed, ncol = 3) +
  scale_fill_manual(values = pal_mix, guide = "none") +
  scale_y_continuous(expand = expansion(mult = c(0, 0.45))) +
  labs(x = NULL, y = "Visits per 4-min observation (mean ± SE)")
save_fig(fig_taxa, "Fig_taxa_by_mix", 190, 130)
print(fig_taxa)

### Fig. – Grupos funcionais: médias do modelo e proporção de polinizadores

In [ ]:
%%R -w 190 -h 75 -u mm -r 150
p_fg_a <- ggplot(emm_fg, aes(mix, response, fill = mix)) +
  geom_col(width = 0.7, colour = "black", linewidth = 0.2) +
  geom_errorbar(aes(ymin = asymp.LCL, ymax = asymp.UCL), width = 0.2, linewidth = 0.3) +
  geom_text(aes(y = asymp.UCL, label = .group), vjust = -0.5, size = 2.4) +
  facet_wrap(~ fg, scales = "free_y") +
  scale_fill_manual(values = pal_mix, guide = "none") +
  scale_y_continuous(expand = expansion(mult = c(0, 0.15))) +
  labs(x = NULL, y = "Visits per observation (model mean, 95% CI)", tag = "a")
p_fg_b <- ggplot(emm_prop, aes(mix, prob, colour = mix)) +
  geom_pointrange(aes(ymin = asymp.LCL, ymax = asymp.UCL), size = 0.3) +
  geom_text(aes(y = asymp.UCL, label = .group), vjust = -0.6, colour = "black", size = 2.4) +
  scale_colour_manual(values = pal_mix, guide = "none") +
  scale_y_continuous(labels = percent, limits = c(NA, 1)) +
  labs(x = NULL, y = "Proportion of pollinators among visitors", tag = "b")
fig_fg <- p_fg_a + p_fg_b + plot_layout(widths = c(2, 1))
save_fig(fig_fg, "Fig_functional_groups", 190, 75)
print(fig_fg)

### Fig. – Índices de diversidade por mix: todos os visitantes, polinizadores e outros visitantes

In [ ]:
%%R -w 190 -h 200 -u mm -r 150
p_lab <- div_fg_tests %>% distinct(group, response, p) %>%
  mutate(lab = ifelse(p < 0.001, "italic(P)<0.001", paste0("italic(P)==\"", sprintf("%.3f", p), "\"")))
fig_div <- ggplot(div_fg_tests, aes(mix, mean, fill = mix)) +
  geom_col(width = 0.7, colour = "black", linewidth = 0.2) +
  geom_errorbar(aes(ymin = mean - se, ymax = mean + se), width = 0.2, linewidth = 0.3) +
  geom_text(aes(y = mean + se, label = .group), vjust = -0.5, size = 2.3) +
  geom_text(data = p_lab, aes(x = 0.5, y = Inf, label = lab), inherit.aes = FALSE, parse = TRUE,
            hjust = 0, vjust = 1.3, size = 2.2) +
  facet_grid(response ~ group, scales = "free_y", switch = "y",
             labeller = labeller(response = as_labeller(sub(" \\(", "\n(", idx_fg)))) +
  scale_fill_manual(values = pal_mix, name = NULL) +
  scale_y_continuous(expand = expansion(mult = c(0, 0.35))) +
  labs(x = NULL, y = NULL) +
  theme(strip.placement = "outside", strip.text.y.left = element_text(angle = 90, size = 7),
        strip.text.x = element_text(face = "bold", size = 7.5), axis.text.x = element_blank(),
        axis.ticks.x = element_blank(), legend.position = "bottom")
save_fig(fig_div, "Fig_diversity_by_group", 190, 200)
print(fig_div)

### Fig. – Dinâmica temporal por grupo funcional (dias após a semeadura)

In [ ]:
%%R -w 190 -h 120 -u mm -r 150
tm_fg <- obs_fg %>% group_by(das, mix, fg) %>%
  summarise(mean = mean(abund), se = sd(abund) / sqrt(n()), .groups = "drop")
temp_das <- obs %>% group_by(das) %>% summarise(temp = mean(temp))
fig_tempo <- ggplot(tm_fg, aes(das, mean, colour = mix)) +
  geom_line(linewidth = 0.4, position = position_dodge(0.6)) +
  geom_pointrange(aes(ymin = mean - se, ymax = mean + se), size = 0.12, linewidth = 0.3,
                  position = position_dodge(0.6)) +
  facet_wrap(~ fg, ncol = 1, scales = "free_y") +
  scale_colour_manual(values = pal_mix, name = lab_mix) +
  scale_x_continuous(breaks = seq(35, 75, 5), limits = c(34, 76)) +
  labs(x = "Days after sowing", y = "Visits per 4-min observation (mean ± SE)") +
  theme(legend.position = "top")
save_fig(fig_tempo, "Fig_temporal_functional_groups", 190, 120)
print(fig_tempo)

### Fig. – Efeitos do mix e do microclima (razões de taxa, estilo Triquet et al. 2024)

In [ ]:
%%R -w 140 -h 70 -u mm -r 150
eff <- irr_all %>% filter(term %in% c("mixMultipurpose", "mixPollinator", "temp_s", "wind_s", "lux_s")) %>%
  mutate(term = recode(term, mixMultipurpose = "Multipurpose vs Carbon", mixPollinator = "Pollinator vs Carbon",
                       temp_s = "Air temperature (+1 SD)", wind_s = "Wind speed (+1 SD)", lux_s = "Light intensity (+1 SD)"),
         term = factor(term, levels = rev(c("Multipurpose vs Carbon", "Pollinator vs Carbon", "Air temperature (+1 SD)",
                                            "Wind speed (+1 SD)", "Light intensity (+1 SD)"))),
         sig = ifelse(p < 0.05, "p < 0.05", "p ≥ 0.05"))
fig_eff <- ggplot(eff, aes(IRR, term, colour = model, shape = sig)) +
  geom_vline(xintercept = 1, linetype = 2, linewidth = 0.3) +
  geom_pointrange(aes(xmin = low, xmax = high), position = position_dodge(0.5), size = 0.3) +
  scale_shape_manual(values = c("p < 0.05" = 16, "p ≥ 0.05" = 1), name = NULL) +
  scale_colour_manual(values = c("Seasonal trend" = "black", "Date as random effect" = "grey55"), name = "Model") +
  scale_x_log10() + labs(x = "Incidence rate ratio (95% CI, log scale)", y = NULL) +
  theme(legend.position = "right")
save_fig(fig_eff, "Fig_effects_forest", 140, 70)
print(fig_eff)

### Fig. – NMDS da comunidade (táxons)

In [ ]:
%%R -w 140 -h 100 -u mm -r 150
hull <- sc_site %>% group_by(mix) %>% slice(chull(NMDS1, NMDS2))
lab_perm <- sprintf("PERMANOVA: R² = %.2f, p = %.3f\nStress = %.2f", pmv$R2[1], pmv$`Pr(>F)`[1], nmds$stress)
vec <- filter(sc_sp, p < 0.05) %>% mutate(lab = lab_taxon_chr(taxon))
fig_nmds <- ggplot(sc_site, aes(NMDS1, NMDS2)) +
  geom_polygon(data = hull, aes(fill = mix), alpha = 0.15, colour = NA) +
  geom_point(aes(colour = mix, shape = block), size = 1.8) +
  geom_segment(data = vec, aes(x = 0, y = 0, xend = NMDS1, yend = NMDS2),
               arrow = arrow(length = unit(1.5, "mm")), linewidth = 0.3, colour = "grey30") +
  geom_text(data = vec, aes(NMDS1 * 1.12, NMDS2 * 1.12, label = lab), parse = TRUE, size = 2.2, colour = "grey20", family = "Arial") +
  annotate("text", x = -Inf, y = Inf, label = lab_perm, hjust = -0.05, vjust = 1.2, size = 2.4) +
  scale_colour_manual(values = pal_mix, name = lab_mix) + scale_fill_manual(values = pal_mix, name = lab_mix) +
  labs(shape = "Block") +
  coord_equal(xlim = range(c(sc_site$NMDS1, sc_sp$NMDS1 * 1.6)), ylim = range(c(sc_site$NMDS2, sc_sp$NMDS2 * 1.25)))
save_fig(fig_nmds, "Fig_NMDS_taxa", 140, 100)
print(fig_nmds)

### Fig. – Curvas de rarefação/extrapolação por tamanho amostral e cobertura

In [ ]:
%%R -w 190 -h 150 -u mm -r 150
fig_inext <- ggiNEXT(inx, type = 1, facet.var = "Order.q", color.var = "Assemblage") +
  scale_colour_manual(values = pal_mix) + scale_fill_manual(values = pal_mix) +
  labs(x = "Number of individuals", y = "Hill number", tag = "a") +
  theme_agee + theme(legend.position = "bottom", legend.title = element_blank())
fig_cov <- ggiNEXT(inx, type = 3, facet.var = "Order.q", color.var = "Assemblage") +
  scale_colour_manual(values = pal_mix) + scale_fill_manual(values = pal_mix) +
  labs(x = "Sample coverage", y = "Hill number", tag = "b") +
  theme_agee + theme(legend.position = "none")
save_fig(fig_inext / fig_cov, "Fig_iNEXT_size_coverage", 190, 150)
print(fig_inext / fig_cov)

### Fig. – Curvas de rank-abundância (Whittaker)

In [ ]:
%%R -w 140 -h 90 -u mm -r 150
rank_ab <- dat %>% group_by(mix, taxon) %>% summarise(n = sum(count), .groups = "drop") %>% filter(n > 0) %>%
  group_by(mix) %>% mutate(rel = n / sum(n), rank = rank(-n, ties.method = "first")) %>% ungroup()
fig_rank <- ggplot(rank_ab, aes(rank, rel, colour = mix)) +
  geom_line(linewidth = 0.4) + geom_point(size = 1.2) +
  scale_y_log10(labels = percent) + scale_x_continuous(breaks = 1:12) +
  scale_colour_manual(values = pal_mix, name = lab_mix) +
  labs(x = "Taxon rank", y = "Relative abundance (log scale)") + theme(legend.position = "top")
save_fig(fig_rank, "Fig_rank_abundance", 140, 90)
print(fig_rank)
# Ordem dos táxons em cada mix (para a legenda da figura)
print(rank_ab %>% arrange(mix, rank) %>% select(mix, rank, taxon, n, rel), n = 40)
save_tab(rank_ab %>% arrange(mix, rank), "T26_rank_abundance")

### Fig. – Partição da diversidade beta entre parcelas de cada mix

In [ ]:
%%R -w 140 -h 80 -u mm -r 150
beta_long <- beta_mix %>% select(mix, sim, nes, bal, gra) %>%
  pivot_longer(-mix) %>% group_by(mix, name) %>% summarise(value = mean(value), .groups = "drop") %>%
  mutate(index = ifelse(name %in% c("sim", "nes"), "Incidence (Sørensen)", "Abundance (Bray-Curtis)"),
         component = recode(name, sim = "Turnover", nes = "Nestedness",
                            bal = "Balanced variation", gra = "Abundance gradient"),
         component = factor(component, levels = c("Nestedness", "Turnover", "Abundance gradient", "Balanced variation")))
fig_beta <- ggplot(beta_long, aes(mix, value, fill = component)) +
  geom_col(width = 0.7, colour = "black", linewidth = 0.2) +
  facet_wrap(~ index) +
  scale_fill_manual(values = c("Turnover" = "#332288", "Nestedness" = "#88CCEE",
                               "Balanced variation" = "#117733", "Abundance gradient" = "#DDCC77"), name = NULL) +
  scale_y_continuous(expand = expansion(mult = c(0, 0.05))) +
  labs(x = NULL, y = "Mean pairwise beta diversity among plots") + theme(legend.position = "top")
save_fig(fig_beta, "Fig_beta_partition", 140, 80)
print(fig_beta)

### Fig. – Rede bipartida mix × táxon, com ícones dos visitantes

In [ ]:
%%R -w 160 -h 120 -u mm -r 150
comp <- dat %>% group_by(mix, taxon, fg) %>% summarise(n = sum(count), .groups = "drop")
ord <- comp %>% group_by(taxon, fg) %>% summarise(n = sum(n), .groups = "drop") %>% arrange(fg, desc(n))
gap <- 0.6   # espaço entre polinizadores e outros visitantes
tx_pos <- ord %>% mutate(k = row_number() + ifelse(fg == "Pollinators", 0, gap),
                         y_t = rescale(-k, c(0, 1)),
                         lab = paste0(lab_taxon_chr(taxon), '~"(', format(n, big.mark = ",", trim = TRUE), ')"'))
mix_tot <- comp %>% group_by(mix) %>% summarise(n = sum(n), .groups = "drop")
mix_pos <- data.frame(mix = levels(dat$mix), y_m = c(0.8, 0.5, 0.2)) %>% left_join(mix_tot, by = "mix")
# Ligações em curva (sigmoide), largura proporcional ao número de visitas
tt <- seq(0, 1, length.out = 40)
edges <- comp %>% filter(n > 0) %>% left_join(mix_pos %>% select(mix, y_m), by = "mix") %>%
  left_join(tx_pos %>% select(taxon, y_t), by = "taxon") %>% mutate(id = row_number())
edge_path <- edges %>% tidyr::crossing(t = tt) %>%
  mutate(x = 0.03 + t * 0.97, y = y_m + (y_t - y_m) * (3 * t^2 - 2 * t^3)) %>% arrange(id, t)
grp_lab <- tx_pos %>% group_by(fg) %>% summarise(y = max(y_t) + 0.045)
x_icon <- 1.07; x_lab <- 1.14
fig_net <- ggplot() +
  geom_path(data = edge_path, aes(x, y, group = id, colour = mix, linewidth = n), alpha = 0.5, lineend = "round") +
  geom_point(data = mix_pos, aes(0, y_m, fill = mix, size = n), shape = 21, colour = "black", stroke = 0.3) +
  geom_point(data = tx_pos, aes(1, y_t), size = 1.3, shape = 21, fill = "white", colour = "grey20", stroke = 0.3) +
  geom_text(data = mix_pos, aes(-0.08, y_m, label = paste0(mix, "\n(", format(n, big.mark = ",", trim = TRUE), " visits)")),
            hjust = 1, size = 2.5, lineheight = 0.9, family = "Arial") +
  geom_text(data = tx_pos, aes(x_lab, y_t, label = lab), parse = TRUE, hjust = 0, size = 2.3, family = "Arial") +
  geom_text(data = grp_lab, aes(x_icon - 0.04, y, label = fg), hjust = 0, size = 2.4, fontface = "bold", family = "Arial") +
  scale_linewidth(range = c(0.15, 4), name = "Visits", breaks = c(50, 250, 500, 1000)) +
  scale_size(range = c(5, 8), guide = "none") +
  scale_colour_manual(values = pal_mix, guide = "none") + scale_fill_manual(values = pal_mix, guide = "none") +
  coord_cartesian(xlim = c(-0.45, 1.75), ylim = c(-0.04, 1.06), clip = "off") +
  theme_void(base_size = 8, base_family = "Arial") +
  theme(legend.position = "bottom", legend.box = "horizontal", plot.margin = margin(4, 4, 4, 4))
# Ícones ao lado de cada táxon (desenho ou foto da pasta icons/)
h_icon <- 0.075
for (i in seq_len(nrow(tx_pos)))
  fig_net <- fig_net + annotation_custom(icon_grob(tx_pos$taxon[i]), xmin = x_icon - 0.05, xmax = x_icon + 0.05,
                                         ymin = tx_pos$y_t[i] - h_icon / 2, ymax = tx_pos$y_t[i] + h_icon / 2)
save_fig(fig_net, "Fig_network", 160, 120)
print(fig_net)
cat("Figuras salvas em out/figs:\n"); print(list.files("out/figs", pattern = "pdf$"))

### Fig. – Correlação entre índices de diversidade e microclima por mix

In [ ]:
%%R -w 190 -h 85 -u mm -r 150
cor_plot <- cor_mix %>%
  mutate(mix = factor(mix, levels = c("Carbon", "Multipurpose", "Pollinator", "All mixes")),
         index = factor(idx_en[index], levels = rev(idx_en)), climate = factor(clim_en[climate], levels = clim_en),
         star = case_when(p_holm < 0.001 ~ "***", p_holm < 0.01 ~ "**", p_holm < 0.05 ~ "*", TRUE ~ ""),
         lab = paste0(sprintf("%.2f", rho), star))
fig_cor <- ggplot(cor_plot, aes(climate, index, fill = rho)) +
  geom_tile(colour = "white", linewidth = 0.4) +
  geom_text(aes(label = lab), size = 2.1) +
  facet_wrap(~ mix, nrow = 1) +
  scale_fill_gradient2(low = "#B2182B", mid = "white", high = "#2166AC", limits = c(-1, 1),   # vermelho = negativa, azul = positiva
                       name = expression("Spearman's"~rho)) +
  labs(x = NULL, y = NULL) +
  theme(axis.text.x = element_text(angle = 45, hjust = 1), axis.line = element_blank(),
        axis.ticks = element_blank(), legend.position = "right")
save_fig(fig_cor, "Fig_correlation_diversity_climate", 190, 85)
print(fig_cor)

### Fig. – RDA: composição dos visitantes × microclima

In [ ]:
%%R -w 140 -h 100 -u mm -r 150
env_lab <- c(temp_s = "Temperature", wind_s = "Wind", lux_s = "Light")
sc_env2 <- sc_env %>% filter(var %in% names(env_lab)) %>% mutate(lab = env_lab[var])
lim <- 0.85 * max(abs(c(sc_obs$RDA1, sc_obs$RDA2)))
m_env <- lim / max(sqrt(sc_env2$RDA1^2 + sc_env2$RDA2^2))
sc_tx2 <- sc_tx %>% mutate(len = sqrt(RDA1^2 + RDA2^2)) %>% slice_max(len, n = 6) %>% mutate(lab = lab_taxon_chr(taxon))
m_tx <- 0.7 * lim / max(sc_tx2$len)
cent <- sc_obs %>% group_by(mix) %>% summarise(RDA1 = mean(RDA1), RDA2 = mean(RDA2))
fig_rda <- ggplot(sc_obs, aes(RDA1, RDA2)) +
  geom_hline(yintercept = 0, linewidth = 0.2, colour = "grey80") + geom_vline(xintercept = 0, linewidth = 0.2, colour = "grey80") +
  geom_point(aes(colour = mix), size = 0.7, alpha = 0.35) +
  geom_segment(data = sc_tx2, aes(x = 0, y = 0, xend = RDA1 * m_tx, yend = RDA2 * m_tx),
               arrow = arrow(length = unit(1.2, "mm")), linewidth = 0.3, colour = "grey45") +
  geom_text(data = sc_tx2, aes(RDA1 * m_tx * 1.15, RDA2 * m_tx * 1.15, label = lab), parse = TRUE,
            size = 2.1, colour = "grey30", family = "Arial") +
  geom_segment(data = sc_env2, aes(x = 0, y = 0, xend = RDA1 * m_env, yend = RDA2 * m_env),
               arrow = arrow(length = unit(1.5, "mm")), linewidth = 0.5, colour = "black") +
  geom_text(data = sc_env2, aes(RDA1 * m_env * 1.12, RDA2 * m_env * 1.12, label = lab), size = 2.4, fontface = "bold") +
  geom_point(data = cent, aes(fill = mix), shape = 23, size = 3, colour = "black") +
  scale_colour_manual(values = pal_mix, name = lab_mix) + scale_fill_manual(values = pal_mix, name = lab_mix) +
  labs(x = paste0("RDA1 (", ax_var[1], "%)"), y = paste0("RDA2 (", ax_var[2], "%)")) +
  coord_equal()
save_fig(fig_rda, "Fig_RDA_climate", 140, 100)
print(fig_rda)

### Fig. – Espécies de plantas de cada mix (com desenho de cada planta)

In [ ]:
%%R -w 150 -h 135 -u mm -r 150
pl <- plants %>%
  mutate(family = factor(family, levels = c("Fabaceae", "Asteraceae", "Brassicaceae", "Polygonaceae", "Poaceae")))
ord_sp <- pl %>% distinct(family, species) %>% arrange(family, species) %>% pull(species)
pl <- pl %>% mutate(species = factor(species, levels = rev(ord_sp)))
pl_n <- plant_sum %>% mutate(lab = paste0(n_species, " species\n", pct_insect_flowers, "% insect-visited"))
pl_icons <- pl %>% distinct(family, species) %>% mutate(icon = as.character(species))
fig_plantas <- ggplot(pl, aes(mix, species)) +
  geom_tile(aes(fill = pct), colour = "white", linewidth = 0.5) +
  geom_text(aes(label = paste0(pct, "%")), size = 2.2) +
  # desenho da planta à esquerda de cada linha (troque por PNG na pasta de ícones, se quiser)
  geom_icon(data = pl_icons, aes(x = 0.15, y = species, icon = icon), size = 5.5, units = "mm") +
  facet_grid(family ~ ., scales = "free_y", space = "free_y", switch = "y") +
  scale_y_discrete(labels = function(x) parse(text = ifelse(grepl(" sp\\.$", x),
    paste0('italic("', sub(" sp\\.$", "", x), '")~"sp."'), paste0('italic("', x, '")')))) +
  scale_x_discrete(position = "top", expand = expansion(add = c(1.15, 0.5)),
                   labels = function(x) paste0(x, "\n", pl_n$lab[match(x, pl_n$mix)])) +
  scale_fill_gradient(low = "#DEEBF7", high = "#08519C", name = "Seed mass\n(% of mix)", limits = c(0, 20)) +
  labs(x = NULL, y = NULL) +
  theme(strip.placement = "outside", strip.text.y.left = element_text(angle = 0, face = "bold", size = 7),
        axis.line = element_blank(), axis.ticks = element_blank(), panel.spacing = unit(0.8, "mm"))
save_fig(fig_plantas, "Fig_cover_crop_species", 150, 135)
print(fig_plantas)

### Fig. – Efeito do microclima em cada índice de diversidade (coeficientes padronizados)

In [ ]:
%%R -w 190 -h 75 -u mm -r 150
ce <- clim_eff %>% filter(term != "hum_z") %>%
  mutate(response = factor(resp_en[response], levels = rev(resp_en)),
         term = factor(c(temp_z = "Air temperature", wind_z = "Wind speed", lux_z = "Light intensity")[term],
                       levels = c("Air temperature", "Wind speed", "Light intensity")),
         dir = case_when(p >= 0.05 ~ "n.s.", beta < 0 ~ "Negative", TRUE ~ "Positive"))
fig_causa <- ggplot(ce, aes(beta, response, colour = dir)) +
  geom_vline(xintercept = 0, linetype = 2, linewidth = 0.3, colour = "grey50") +
  geom_pointrange(aes(xmin = low, xmax = high), size = 0.25, linewidth = 0.4) +
  facet_wrap(~ term, nrow = 1) +
  scale_colour_manual(values = c(Negative = "#B2182B", Positive = "#2166AC", n.s. = "grey60"), name = NULL) +
  labs(x = "Standardised effect (β ± 95% CI)", y = NULL) +
  theme(legend.position = "bottom", panel.border = element_rect(fill = NA, colour = "grey80", linewidth = 0.3))
save_fig(fig_causa, "Fig_climate_effects_diversity", 190, 75)
print(fig_causa)

### Fig. – Diagrama de caminhos: estação → microclima → abundância → diversidade

In [ ]:
%%R -w 190 -h 100 -u mm -r 150
stars <- function(p) ifelse(p < 0.001, "***", ifelse(p < 0.01, "**", ifelse(p < 0.05, "*", "")))
nodes <- data.frame(id = c("das", "temp_z", "wind_z", "lux_z", "logN_z", "q0_z", "q1_z"),
                    lab = c("Days after\nsowing", "Air\ntemperature", "Wind\nspeed", "Light\nintensity",
                            "Visitor\nabundance", "Richness\n(q = 0)", "Shannon\ndiversity (q = 1)"),
                    x = c(-0.2, 1.6, 1.6, 1.6, 3.2, 4.8, 4.8), y = c(2, 3.2, 2, 0.8, 2, 3.2, 0.8)) %>%
  mutate(lab = ifelse(id %in% names(r2_sem), paste0(lab, "\nR\u00b2m = ", sprintf("%.2f", r2_sem[id])), lab))
# Estação → microclima: efeito linear e quadrático juntos numa seta (preta); demais setas: azul = positivo, vermelho = negativo
season <- paths %>% filter(from %in% c("das_z", "das2_z"), to %in% c("temp_z", "wind_z", "lux_z")) %>%
  group_by(to) %>%
  summarise(lab = paste0(sprintf("%.2f", beta[from == "das_z"]), stars(p[from == "das_z"]), " (linear)\n",
                         sprintf("%.2f", beta[from == "das2_z"]), stars(p[from == "das2_z"]), " (quadratic)"),
            sig = any(p < 0.05), beta = max(abs(beta)), from = "das", .groups = "drop") %>% mutate(col = "Seasonal trend")
pth <- paths %>% filter(!from %in% c("das_z", "das2_z")) %>%
  mutate(sig = p < 0.05, col = ifelse(beta < 0, "Negative", "Positive"), lab = paste0(sprintf("%.2f", beta), stars(p))) %>%
  bind_rows(season) %>% filter(sig) %>%
  left_join(nodes %>% select(from = id, x0 = x, y0 = y), by = "from") %>%
  left_join(nodes %>% select(to = id, x1 = x, y1 = y), by = "to") %>%
  mutate(len = sqrt((x1 - x0)^2 + (y1 - y0)^2), sh = 0.5 / len,
         xs = x0 + (x1 - x0) * sh, ys = y0 + (y1 - y0) * sh, xe = x1 - (x1 - x0) * sh, ye = y1 - (y1 - y0) * sh,
         xm = xs + (xe - xs) * 0.45, ym = ys + (ye - ys) * 0.45 + ifelse(abs(ye - ys) < 0.01, ifelse(col == "Seasonal trend", -0.3, 0.2), 0))
fig_sem <- ggplot() +
  geom_segment(data = pth, aes(xs, ys, xend = xe, yend = ye, colour = col, linewidth = abs(beta)),
               arrow = arrow(length = unit(1.6, "mm"), type = "closed")) +
  geom_label(data = pth, aes(xm, ym, label = lab, colour = col), size = 2, label.size = 0, lineheight = 0.9,
             label.padding = unit(0.4, "mm"), fill = "white", show.legend = FALSE) +
  geom_label(data = nodes, aes(x, y, label = lab), size = 2.3, label.r = unit(1, "mm"), label.size = 0.3, lineheight = 0.9) +
  scale_colour_manual(values = c(Negative = "#B2182B", Positive = "#2166AC", `Seasonal trend` = "grey25"), name = NULL) +
  scale_linewidth(range = c(0.4, 1.6), guide = "none") +
  coord_cartesian(xlim = c(-0.6, 5.3), ylim = c(0.4, 3.6)) +
  labs(caption = sprintf(paste("Only significant paths (p < 0.05) are shown; values are standardised coefficients.",
                               "Fisher's C = %.2f, df = %d, p = %.2f.\nBlock and days after sowing (linear and quadratic) are in every equation;",
                               "mix is also in the abundance and diversity equations. R\u00b2m: marginal R\u00b2."),
                         sem_fit$Fisher_C, sem_fit$df, sem_fit$p)) +
  theme_void(base_size = 8, base_family = "Arial") +
  theme(legend.position = "bottom", plot.caption = element_text(size = 6, hjust = 0))
save_fig(fig_sem, "Fig_path_model", 190, 100)
print(fig_sem)

### Fig. – Biomassa das plantas de cobertura e diversidade dos polinizadores

In [ ]:
%%R -w 190 -h 110 -u mm -r 150
p_b1 <- ggplot(bio, aes(mix, biomass, colour = mix)) +
  geom_jitter(width = 0.12, size = 1, alpha = 0.6) +
  geom_pointrange(data = emm_bio, aes(y = emmean, ymin = lower.CL, ymax = upper.CL), colour = "black", size = 0.3) +
  geom_text(data = emm_bio, aes(y = upper.CL, label = .group), colour = "black", vjust = -0.6, size = 2.4) +
  scale_colour_manual(values = pal_mix, guide = "none") +
  scale_y_continuous(expand = expansion(mult = c(0.05, 0.12))) +
  labs(x = NULL, y = expression("Fresh biomass (Mg ha"^-1*")"), tag = "a") +
  theme(axis.text.x = element_text(angle = 30, hjust = 1))
bio_long <- plot_div %>% select(plot_id, mix, biomass, N, q0, q1, prop_poll) %>%
  pivot_longer(c(N, q0, q1, prop_poll), names_to = "index") %>%
  left_join(bio_tests %>% select(index, p_biomass, rho, p_rho), by = "index") %>%
  mutate(index_lab = factor(idx_plot[index], levels = idx_plot[c("N", "q0", "q1", "prop_poll")]))
ann <- bio_long %>% distinct(index_lab, rho, p_rho) %>%
  mutate(txt = sprintf("rho = %.2f; p = %s", rho, ifelse(p_rho < 0.001, "< 0.001", sprintf("%.3f", p_rho))))
p_b2 <- ggplot(bio_long, aes(biomass, value)) +
  geom_smooth(aes(linetype = p_biomass < 0.05), method = "lm", formula = y ~ x, colour = "black",
              linewidth = 0.4, se = TRUE, fill = "grey85") +
  geom_point(aes(colour = mix), size = 1.2) +
  geom_text(data = ann, aes(x = -Inf, y = Inf, label = txt), hjust = -0.05, vjust = 1.3, size = 2) +
  facet_wrap(~ index_lab, scales = "free_y", nrow = 2) +
  scale_colour_manual(values = pal_mix, name = lab_mix) +
  scale_linetype_manual(values = c(`TRUE` = 1, `FALSE` = 2), guide = "none") +
  labs(x = expression("Fresh biomass (Mg ha"^-1*")"), y = "Value per plot", tag = "b") +
  theme(legend.position = "bottom")
fig_biomassa <- p_b1 + p_b2 + plot_layout(widths = c(1, 2.6))
save_fig(fig_biomassa, "Fig_biomass_diversity", 190, 110)
print(fig_biomassa)

## Salvar resultados no Google Drive

In [ ]:
pasta_tab = subpasta(PASTA_DADOS, "Resultados_tabelas_v2")
for f in sorted(glob.glob("/content/out/data/*.csv")):
    enviar(f, PASTA_DADOS); print("02_Dados <-", os.path.basename(f))
for f in sorted(glob.glob("/content/out/tables/*.csv")):
    enviar(f, pasta_tab); print("02_Dados/Resultados_tabelas_v2 <-", os.path.basename(f))
for f in sorted(glob.glob("/content/out/figs/*")):
    enviar(f, PASTA_FIGURAS); print("04_Figuras <-", os.path.basename(f))
print("Concluído.")

## Documento de resultados com figuras e tabelas
- Lê o texto do guia (`resultados_estatisticos_v2.md`, em 01_Manuscrito) e insere, em cada seção, as figuras e as tabelas geradas acima.
- As tabelas seguem o padrão Elsevier/AGEE: título acima, só linhas horizontais (sem linhas verticais nem sombreamento), médias com letras sobrescritas (a = maior média) e notas abaixo.
- Salva em 01_Manuscrito dois Google Docs (substituindo os de mesmo nome): o guia completo e um arquivo só com as tabelas (**Tables – AGEE format (v2)**).

In [ ]:
!pip -q install python-docx
# Tabelas no padrão de artigo (Elsevier/AGEE): texto editável, título acima, notas abaixo,
# só linhas horizontais (acima e abaixo do cabeçalho e no fim), sem linhas verticais nem sombreamento.
# Marcação nas células: *texto* = itálico, ^texto^ = sobrescrito (letras de comparação).
import re
import numpy as np
import pandas as pd
from docx.shared import Pt
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml import OxmlElement
from docx.oxml.ns import qn

MIXES = ["Carbon", "Multipurpose", "Pollinator"]
TERM = {"mix": "Mix", "temp_s": "Air temperature", "temp_z": "Air temperature", "temp": "Air temperature",
        "wind_s": "Wind speed", "wind_z": "Wind speed", "wind": "Wind speed",
        "lux_s": "Light intensity", "lux_z": "Light intensity", "lux": "Light intensity",
        "hum_s": "Relative humidity", "hum_z": "Relative humidity", "hum": "Relative humidity",
        "poly(day_s, 2)": "Days after sowing (linear + quadratic)", "block": "Block", "fg": "Functional group",
        "mix:fg": "Mix × functional group", "biomass_s": "Cover crop biomass", "biomass": "Cover crop biomass",
        "system": "Cropping system", "Residual": "Residual", "Residuals": "Residual", "Total": "Total",
        "das_z": "Days after sowing", "das2_z": "Days after sowing²", "logN_z": "Visitor abundance (log)",
        "q0_z": "Richness (q = 0)", "q1_z": "Shannon diversity (q = 1)", "q2_z": "Simpson diversity (q = 2)",
        "pielou_z": "Pielou evenness (J′)", "prop_poll_z": "Proportion of pollinators"}
INDEX = {"N": "Visits per plot", "q0": "Richness (q = 0)", "q1": "Shannon diversity (q = 1)",
         "q2": "Simpson diversity (q = 2)", "shannon": "Shannon index (H′)", "simpson": "Simpson index (1 − D)",
         "pielou": "Pielou evenness (J′)", "berger_parker": "Berger–Parker dominance",
         "prop_poll": "Proportion of pollinators"}
FG = {"Pollinators": "Pollinators", "Other flower visitors": "Other flower visitors"}


def sci(t):
    """Nomes científicos em itálico (gênero e espécie; 'sp.' e morfotipos ficam em redondo)."""
    t = str(t)
    if re.match(r"^[A-Z][a-z]+ [a-z]+$", t) and not t.endswith(" sp."):
        return f"*{t}*"
    m = re.match(r"^([A-Z][a-z]+) (sp\.)$", t)
    return f"*{m.group(1)}* sp." if m else t


def pval(p):
    if p is None or (isinstance(p, float) and np.isnan(p)): return "–"
    return "< 0.001" if p < 0.001 else f"{p:.3f}"


def num(x, d=2):
    if x is None or (isinstance(x, float) and np.isnan(x)): return "–"
    return f"{x:.{d}f}".replace("-", "−")


def stars(p):
    return "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""


def ci(lo, hi, d=1):
    return f"{num(lo, d)}–{num(hi, d)}"


def emm_cols(d):
    """Nomes das colunas de média e IC do emmeans (escala da resposta ou do modelo)."""
    m = next(c for c in ("response", "prob", "rate", "emmean") if c in d.columns)
    lo = next(c for c in ("asymp.LCL", "lower.CL") if c in d.columns)
    hi = next(c for c in ("asymp.UCL", "upper.CL") if c in d.columns)
    return m, lo, hi


def mean_letter(d, dec=1, pct=False):
    m, lo, hi = emm_cols(d)
    f = (lambda v: v * 100) if pct else (lambda v: v)
    return d.apply(lambda r: f"{num(f(r[m]), dec)}^{str(r['.group']).strip()}^ ({ci(f(r[lo]), f(r[hi]), dec)})", axis=1)


def anova(d, chisq=True):
    d = d.copy()
    stat = "Chisq" if "Chisq" in d.columns else "F value" if "F value" in d.columns else "F"
    pcol = next(c for c in d.columns if c.startswith("Pr("))
    out = pd.DataFrame({"Source of variation": d["term"].map(lambda t: TERM.get(t, t))})
    if "Sum Sq" in d.columns: out["SS"] = d["Sum Sq"].map(lambda v: num(v, 1))
    out["df"] = d["Df"].map(lambda v: "–" if pd.isna(v) else str(int(v)))
    out["χ²" if stat == "Chisq" else "*F*"] = d[stat].map(lambda v: num(v, 2))
    out["*P*"] = d[pcol].map(pval)
    return out


def permanova(d):
    return pd.DataFrame({"Source of variation": d["term"].map(lambda t: TERM.get(t, t)),
                         "df": d["Df"].map(lambda v: str(int(v))),
                         "SS": d["SumOfSqs"].map(lambda v: num(v, 3)),
                         "*R*²": d["R2"].map(lambda v: num(v, 2)),
                         "Pseudo-*F*": d["F"].map(lambda v: num(v, 2)),
                         "*P*": d["Pr(>F)"].map(pval)})


# ---------- um formatador por tabela: recebe o CSV e devolve (tabela, nota de rodapé) ----------
def f_T1(d):
    rows = []
    for g, sub in d.groupby("fg", sort=False):
        rows.append({"Taxon": f"**{FG.get(g, g)}**", **{m: "" for m in MIXES}, "Total": "", "%": ""})
        for _, r in sub.iterrows():
            rows.append({"Taxon": "  " + sci(r["taxon"]), **{m: f"{int(r[m]):,}" for m in MIXES},
                         "Total": f"{int(r['Total']):,}", "%": num(r["Percent"], 1)})
    rows.append({"Taxon": "Total", **{m: f"{int(d[m].sum()):,}" for m in MIXES},
                 "Total": f"{int(d['Total'].sum()):,}", "%": "100.0"})
    return pd.DataFrame(rows), "Sum of 13 surveys of 4 min in eight plots per mix."

def f_T2(d):
    out = []
    for g in d["fg"].unique():
        sub = d[d["fg"] == g].set_index("mix")
        out.append({"Functional group": FG.get(g, g), **{m: f"{int(sub.loc[m, 'n']):,} ({num(sub.loc[m, 'percent'], 1)}%)" for m in MIXES}})
    tot = d.groupby("mix")["n"].sum()
    out.append({"Functional group": "Total", **{m: f"{int(tot[m]):,} (100.0%)" for m in MIXES}})
    return pd.DataFrame(out), "Values are numbers of visits, with the percentage within each mix in parentheses."

def f_anova_glmm(d):
    return anova(d), "Type II Wald χ² tests from a negative binomial GLMM with plot as random effect."

def f_T5(d):
    d = d.set_index("mix").loc[MIXES].reset_index()
    return pd.DataFrame({"Mix": d["mix"], "Visits per observation (95% CI)": mean_letter(d)}), \
        "Model-estimated means. Means followed by the same letter do not differ (Šidák-adjusted comparisons; a = highest mean)."

def f_T8(d):
    out = []
    for m in MIXES:
        row = {"Mix": m}
        for g in d["fg"].unique():
            r = d[(d["mix"] == m) & (d["fg"] == g)]
            row[FG.get(g, g)] = mean_letter(r).iloc[0]
        out.append(row)
    return pd.DataFrame(out), "Model-estimated visits per observation (95% CI). Within each functional group, means followed by the same letter do not differ (Šidák; a = highest mean)."

def f_T9(d):
    d = d.set_index("mix").loc[MIXES].reset_index()
    return pd.DataFrame({"Mix": d["mix"], "Pollinators among all visitors, % (95% CI)": mean_letter(d, 1, pct=True)}), \
        "Beta-binomial GLMM. Means followed by the same letter do not differ (Šidák; a = highest mean)."

def f_T10a(d):
    return pd.DataFrame({"Taxon": d["taxon"].map(sci), "Individuals": d["n"].map(lambda v: f"{int(v):,}"),
                         "Surveys with records": d["n_evals"].astype(int).astype(str),
                         "Blocks with records": d["n_blocks"].astype(int).astype(str)}), \
        "Taxa with ≥ 100 individuals, records in ≥ 5 surveys and in all four blocks were modelled individually."

def f_T10(d):
    d = d[d["term"] == "mix"]
    return pd.DataFrame({"Taxon": d["taxon"].map(sci), "df": d["Df"].astype(int).astype(str), "χ²": d["Chisq"].map(num),
                         "*P*": d["Pr(>Chisq)"].map(pval)}), \
        "Effect of mix in taxon-level negative binomial GLMMs (Type II Wald χ²), adjusted for microclimate, seasonal trend and block."

def f_T11(d):
    out = []
    for t in d["taxon"].unique():
        row = {"Taxon": sci(t)}
        for m in MIXES:
            r = d[(d["taxon"] == t) & (d["mix"] == m)]
            row[m] = mean_letter(r, 2).iloc[0] if len(r) else "–"
        out.append(row)
    return pd.DataFrame(out), "Model-estimated visits per observation (95% CI). Within each taxon, means followed by the same letter do not differ (Šidák; a = highest mean)."

def f_perm(d):
    return permanova(d), "Hellinger-transformed abundances, Bray–Curtis distance; 9999 permutations restricted within blocks."

def f_T14(d):
    return pd.DataFrame({"Comparison": d["pair"].str.replace(" vs ", " vs. "), "*R*²": d["R2"].map(num),
                         "Pseudo-*F*": d["F"].map(num), "*P*": d["p"].map(pval), "*P* (Holm)": d["p_holm"].map(pval)}), "9999 permutations restricted within blocks; P adjusted by the Holm method."

def f_T16(d):
    scols = [c for c in d.columns if c.startswith("s.")]
    grp = d.apply(lambda r: " + ".join(c[2:] for c in scols if r[c] == 1), axis=1) if scols else d.get("index", "")
    d = d.assign(grp=grp).sort_values("p.value")
    return pd.DataFrame({"Taxon": d["taxon"].map(sci), "Mix": d["grp"], "IndVal": d["stat"].map(num),
                         "*P*": d["p.value"].map(pval), "*P* (Holm)": d["p_holm"].map(pval)}), \
        "IndVal.g with 9999 permutations restricted within blocks."

def f_T18(d):
    out = []
    for v in d["response"].unique():
        sub = d[d["response"] == v].set_index("mix")
        row = {"Index": INDEX.get(v, v)}
        for m in MIXES:
            row[m] = f"{num(sub.loc[m, 'emmean'])}^{str(sub.loc[m, '.group']).strip()}^ ± {num(sub.loc[m, 'SE'])}"
        row["*F*"] = num(sub["F"].iloc[0]); row["*P*"] = pval(sub["p"].iloc[0])
        out.append(row)
    return pd.DataFrame(out), "Estimated mean ± SE per plot (n = 8 per mix; linear model with block). Means followed by the same letter do not differ (Šidák; a = highest mean)."

def f_T20(d):
    a = "Assemblage" if "Assemblage" in d.columns else "site" if "site" in d.columns else d.columns[0]
    return pd.DataFrame({"Mix": d[a], "Individuals (*n*)": d["n"].map(lambda v: f"{int(v):,}"),
                         "Observed richness": d["S.obs"].astype(int).astype(str),
                         "Sample coverage (%)": d["SC"].map(lambda v: num(100 * v, 1))}), "Sample coverage estimated with iNEXT (Chao et al. 2014)."

def f_T21(d):
    a = "Assemblage" if "Assemblage" in d.columns else "site" if "site" in d.columns else d.columns[0]
    q = "Order.q" if "Order.q" in d.columns else "order"
    lo = "qD.LCL" if "qD.LCL" in d.columns else "qD.95.LCL"; hi = "qD.UCL" if "qD.UCL" in d.columns else "qD.95.UCL"
    out = []
    for m in d[a].unique():
        row = {"Mix": m}
        for qq, lab in [(0, "Richness (q = 0)"), (1, "Shannon (q = 1)"), (2, "Simpson (q = 2)")]:
            r = d[(d[a] == m) & (d[q] == qq)]
            row[lab] = f"{num(r['qD'].iloc[0])} ({ci(r[lo].iloc[0], r[hi].iloc[0], 2)})" if len(r) else "–"
        out.append(row)
    sc = d["SC"].iloc[0] if "SC" in d.columns else np.nan
    return pd.DataFrame(out), f"Hill numbers (95% CI) standardised to a sample coverage of {num(100 * sc, 1)}%."

def f_T22(d):
    cols = [("sor", "Sørensen"), ("sim", "Turnover"), ("nes", "Nestedness"),
            ("bc", "Bray–Curtis"), ("bal", "Balanced variation"), ("gra", "Abundance gradient")]
    out = pd.DataFrame({"Mix": d["mix"]})
    for k, lab in cols:
        out[lab] = d.apply(lambda r: f"{num(r[k + '_mean'])} ± {num(r[k + '_se'])}", axis=1)
    return out, "Mean ± SE of pairwise dissimilarities among the eight plots of each mix. Sørensen = turnover + nestedness (incidence); Bray–Curtis = balanced variation + abundance gradient (abundance)."

def f_T25(d):
    g = d.groupby("mix")
    se = lambda x: x.std() / np.sqrt(len(x))
    return pd.DataFrame({"Mix": list(g.groups.keys()),
                         "Coefficient of variation": [f"{num(x.mean())} ± {num(se(x))}" for _, x in g["cv"]],
                         "Taxon synchrony": [f"{num(x.mean())} ± {num(se(x))}" for _, x in g["synchrony"]]}), \
        "Mean ± SE per plot across the 13 surveys (n = 8 plots per mix)."

def f_T27(d):
    d = d[(d["p_holm"] < 0.05) | (d["p_partial_holm"] < 0.05)]
    clim = {"temp": "Air temperature", "hum": "Relative humidity", "wind": "Wind speed", "lux": "Light intensity"}
    idx = {**INDEX, "N": "Abundance"}
    return pd.DataFrame({"Mix": d["mix"], "Index": d["index"].map(lambda v: idx.get(v, v)),
                         "Variable": d["climate"].map(clim), "*n*": d["n"].astype(int).astype(str),
                         "ρ": d["rho"].map(num), "*P* (Holm)": d["p_holm"].map(pval),
                         "Partial ρ": d["rho_partial"].map(num), "*P* partial (Holm)": d["p_partial_holm"].map(pval)}), \
        "Only pairs significant after Holm correction are shown. Partial ρ controls for days after sowing."

def f_rda(d):
    return pd.DataFrame({"Variable": d["term"].map(lambda t: TERM.get(t, t)), "df": d["Df"].astype(int).astype(str),
                         "Variance": d["Variance"].map(lambda v: num(v, 4)), "*F*": d["F"].map(num),
                         "*P*": d["Pr(>F)"].map(pval)}), "999 permutations restricted within plots."

def f_T30(d):
    return pd.DataFrame({"Fraction": d["fraction"], "Adjusted *R*² (%)": d["adj_R2"].map(lambda v: num(max(100 * v, 0), 1))}), \
        "Fractions from varpart (adjusted R²); negative values are shown as zero. Time: days after sowing (linear + quadratic)."

def f_T23(d):
    lab = {"connectance": "Connectance", "H2": "Specialisation (*H*₂′)", "weighted NODF": "Nestedness (weighted NODF)",
           "Shannon diversity": "Shannon diversity of interactions"}
    return pd.DataFrame({"Metric": d["metric"].map(lambda v: lab.get(v, v)), "Value": d["value"].map(num)}), \
        "Mix–visitor network built from the total visits of each taxon in each mix (bipartite package)."

def f_T24(d):
    out = pd.DataFrame({"Taxon": d["taxon"].map(sci)})
    if "degree" in d.columns: out["Degree"] = d["degree"].astype(int).astype(str)
    if "d" in d.columns: out["*d*′"] = d["d"].map(num)
    return out, None

def f_T32(d):
    return pd.DataFrame({"Mix": d["mix"], "Species": d["n_species"].astype(str), "Families": d["n_families"].astype(str),
                         "Insect-visited species": d["n_insect_flowers"].astype(str),
                         "Insect-visited seed mass (%)": d["pct_insect_flowers"].astype(int).astype(str)}), \
        "Insect-visited species: all families except Poaceae (wind-pollinated grasses)."

def f_T33(d):
    out = []
    for r_ in d["response"].unique():
        sub = d[d["response"] == r_].set_index("term")
        row = {"Response": TERM.get(r_, r_)}
        for t, lab in [("temp_z", "Air temperature"), ("wind_z", "Wind speed"), ("lux_z", "Light intensity")]:
            row[lab] = f"{num(sub.loc[t, 'beta'])}{stars(sub.loc[t, 'p']).replace('*', chr(92) + '*')} ({num(sub.loc[t, 'se'])})"
        row["*R*²m"] = num(sub["R2_marginal"].iloc[0])
        row["Microclimate χ² (*P*)"] = f"{num(sub['LRT_climate_chisq'].iloc[0])} ({pval(sub['LRT_climate_p'].iloc[0])})"
        out.append(row)
    return pd.DataFrame(out), "Standardised coefficients (SE) from linear mixed models with mix, block, days after sowing (linear + quadratic) and plot as random effect. Microclimate χ²: likelihood-ratio test of the three variables together (df = 3). \\* *P* < 0.05; \\*\\* *P* < 0.01; \\*\\*\\* *P* < 0.001."

def f_T34(d):
    out, last = [], None
    for _, r in d.iterrows():
        out.append({"Response": TERM.get(r["to"], r["to"]) if r["to"] != last else "",
                    "Predictor": TERM.get(r["from"], r["from"]), "β": num(r["beta"]), "SE": num(r["se"]),
                    "*P*": pval(r["p"]), "*R*²m": num(r["R2_marginal_response"]) if r["to"] != last else ""})
        last = r["to"]
    return pd.DataFrame(out), "Standardised path coefficients. Block (all equations) and mix (abundance and diversity equations) were included as covariates; their coefficients are not shown."

def f_T35(d):
    cl = d[d["claim"] != "Fisher's C"]; fc = d[d["claim"] == "Fisher's C"].iloc[0]
    out = pd.DataFrame({"Independence claim": cl["claim"].map(lambda c: " → ".join(TERM.get(x.strip(), x.strip()) for x in c.split("->"))) + " | season, block",
                        "*P*": cl["p"].map(pval)})
    return out, f"Fisher's C = {num(fc['C'])}, df = {int(fc['df'])}, P = {pval(fc['p'])}; P > 0.05 indicates that the path model is consistent with the data."

def f_T36(d):
    return pd.DataFrame({"Response": d["response"].map(lambda v: TERM.get(v, v)), "Microclimate": d["climate"].map(lambda v: TERM.get(v, v)),
                         "Direct β (*P*)": d.apply(lambda r: f"{num(r['direct'])} ({pval(r['p_direct'])})", axis=1),
                         "Indirect β via abundance (*P*)": d.apply(lambda r: f"{num(r['indirect'], 3)} ({pval(r['p_indirect'])})", axis=1),
                         "Total β": d["total"].map(num)}), "Indirect effects are products of path coefficients; P from the delta method."

def f_T38(d):
    d = d.set_index("mix").loc[MIXES].reset_index()
    return pd.DataFrame({"Mix": d["mix"], "Fresh biomass, Mg ha⁻¹ (95% CI)": mean_letter(d)}), \
        "Means followed by the same letter do not differ (Šidák; a = highest mean)."

def f_T39(d):
    return pd.DataFrame({"Index": d["index"].map(lambda v: INDEX.get(v, v)),
                         "Slope": d["slope_per_Mg"].map(lambda v: num(v, 3)), "*P*": d["p_biomass"].map(pval),
                         "Slope (adj.)": d["slope_adj_mix"].map(lambda v: num(v, 3)), "*P* (adj.)": d["p_biomass_adj_mix"].map(pval),
                         "Spearman ρ": d["rho"].map(num), "*P* (ρ)": d["p_rho"].map(pval)}), \
        "Slope: change per Mg ha⁻¹ of fresh biomass in a linear model with block. Adj.: model also including mix and cropping system (n = 24 plots)."

def f_T40(d):
    out = anova(d); out.insert(0, "Model", [m if i == 0 or m != p else "" for i, (m, p) in enumerate(zip(d["model"], [None] + list(d["model"][:-1])))])
    return out, "Type II Wald χ² tests; negative binomial GLMM with plot as random effect."

def f_T43(d):
    out = []; prev = None
    for g in ["All flower visitors", "Pollinators", "Other flower visitors"]:
        for v in [x for x in ["N", "q0", "q1", "q2", "pielou"] if x in set(d["response"])]:
            sub = d[(d["group"] == g) & (d["response"] == v)].set_index("mix")
            if sub.empty: continue
            row = {"Group": g if g != prev else "", "Index": INDEX.get(v, v)}; prev = g
            dec = 1 if v == "N" else 2
            for m in MIXES:
                row[m] = f"{num(sub.loc[m, 'mean'], dec)}^{str(sub.loc[m, '.group']).strip()}^ ± {num(sub.loc[m, 'se'], dec)}"
            r0 = sub.iloc[0]
            row["*F* (df)"] = f"{num(r0['F'])} ({int(r0['df'])}, {int(r0['df_res'])})"; row["*P*"] = pval(r0["p"])
            out.append(row)
    return pd.DataFrame(out), ("Mean ± SE per plot (n = 8 per mix). Indices computed with all flower-visiting taxa, only pollinators, "
                               "or only other flower visitors. Linear model with block; means followed by the same letter do not differ "
                               "(Šidák; a = highest mean). Pielou evenness excludes plots with fewer than two taxa.")

FORMAT = {"T1_abundance_taxon_mix": f_T1, "T2_abundance_functional_group_mix": f_T2,
          "T3_anova_total_abundance": f_anova_glmm, "T4_anova_total_abundance_date_random": f_anova_glmm,
          "T5_emmeans_total_abundance": f_T5, "T7_anova_functional_group": f_anova_glmm,
          "T8_emmeans_functional_group": f_T8, "T9_emmeans_proportion_pollinators": f_T9,
          "T10a_taxon_occurrence": f_T10a, "T10_anova_by_taxon": f_T10, "T11_emmeans_by_taxon": f_T11,
          "T12_permanova_all_taxa": f_perm, "T13_permanova_Pollinators": f_perm,
          "T13_permanova_Other_flower_visitors": f_perm, "T14_permanova_pairwise": f_T14, "T16_indval": f_T16,
          "T18_alpha_diversity_tests": f_T18, "T20_inext_sample_coverage": f_T20, "T21_inext_equal_coverage": f_T21,
          "T22_beta_partition_within_mix": f_T22, "T25_temporal_stability": f_T25,
          "T27_spearman_diversity_climate_by_mix": f_T27, "T28_rda_climate_margin": f_rda,
          "T29_rda_climate_after_season": f_rda, "T30_variance_partitioning": f_T30, "T23_network_metrics": f_T23,
          "T24_network_taxon_specialization": f_T24, "T32_cover_crop_mix_summary": f_T32,
          "T33_climate_effects_on_diversity": f_T33, "T34_path_coefficients": f_T34, "T35_path_model_fit": f_T35,
          "T36_direct_indirect_climate_effects": f_T36, "T37_anova_biomass": lambda d: (anova(d), "Type II ANOVA (linear model)."),
          "T38_emmeans_biomass": f_T38, "T39_biomass_diversity_plot_level": f_T39,
          "T40_abundance_model_with_biomass": f_T40, "T43_alpha_diversity_by_group_tests": f_T43, "T41_permanova_biomass": lambda d: (permanova(d), "Marginal tests; 9999 permutations restricted within blocks.")}


def formatar(name, d):
    """Aplica o formatador da tabela; se o CSV vier com colunas inesperadas, usa uma limpeza genérica."""
    try:
        return FORMAT[name](d)
    except Exception as e:
        print(f"Aviso: formatação padrão em {name} ({e})")
        d = d.copy()
        for c in d.columns:
            if d[c].dtype.kind == "f":
                d[c] = d[c].map(pval if re.search(r"^p$|^p_|Pr\(|p\.value", c, re.I) else num)
        return d.astype(str), None


# ---------- escrita no Word ----------
def _borda(cell, lado, sz):
    tcPr = cell._tc.get_or_add_tcPr()
    b = tcPr.find(qn("w:tcBorders"))
    if b is None:
        b = OxmlElement("w:tcBorders"); tcPr.append(b)
    e = OxmlElement(f"w:{lado}")
    e.set(qn("w:val"), "single"); e.set(qn("w:sz"), str(sz)); e.set(qn("w:color"), "000000")
    b.append(e)


def _texto(par, txt, size, bold=False):
    # **negrito**, *itálico*, ^sobrescrito^; \* = asterisco literal
    txt = str(txt).replace("\\*", "\u2217")
    for part in re.split(r"(\*\*[^*]+\*\*|\*[^*]+\*|\^[^^]+\^)", txt):
        if not part: continue
        if part.startswith("**"): r = par.add_run(part[2:-2]); r.bold = True
        elif part.startswith("*"): r = par.add_run(part[1:-1]); r.italic = True
        elif part.startswith("^"): r = par.add_run(part[1:-1]); r.font.superscript = True
        else: r = par.add_run(part.replace("\u2217", "*"))
        if bold: r.bold = True
        r.font.size = Pt(size)


def tabela_word(doc, n, caption, tab, note=None, size=8):
    """Título acima, três linhas horizontais, nota abaixo."""
    cap = doc.add_paragraph(); cap.paragraph_format.keep_with_next = True
    r = cap.add_run(f"Table {n}. "); r.bold = True; r.font.size = Pt(9)
    _texto(cap, caption, 9)
    t = doc.add_table(rows=1, cols=len(tab.columns))
    t.style = doc.styles["Normal Table"] if "Normal Table" in [s.name for s in doc.styles] else None
    for i, c in enumerate(tab.columns):
        cell = t.rows[0].cells[i]; cell.text = ""
        _texto(cell.paragraphs[0], str(c).strip(), size)
        _borda(cell, "top", 8); _borda(cell, "bottom", 4)
    for _, row in tab.iterrows():
        cells = t.add_row().cells
        for i, c in enumerate(tab.columns):
            cells[i].text = ""
            _texto(cells[i].paragraphs[0], str(row[c]), size)
    for cell in t.rows[-1].cells:
        _borda(cell, "bottom", 8)
    for row in t.rows:
        for i, cell in enumerate(row.cells):
            p = cell.paragraphs[0]
            p.paragraph_format.space_after = Pt(1); p.paragraph_format.space_before = Pt(1)
            p.alignment = WD_ALIGN_PARAGRAPH.LEFT if i == 0 else WD_ALIGN_PARAGRAPH.CENTER
    if note:
        p = doc.add_paragraph(); _texto(p, note, size - 0.5)
        p.paragraph_format.space_after = Pt(10)
    else:
        doc.add_paragraph()

# Monta o documento de resultados (.docx): texto do guia + figuras + tabelas de cada seção
import os, re, glob
import pandas as pd
from docx import Document
from docx.shared import Pt, Cm, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH

def gerar_relatorio(md_path, figs_dir, tabs_dir, out_path, tables_path=None):
    # Figuras e tabelas de cada seção do guia: (tipo, arquivo, legenda em inglês, largura em cm)
    assets = {
        1: [("tab", "T1_abundance_taxon_mix", "Flower visits per taxon in the three cover crop mixes."),
            ("tab", "T2_abundance_functional_group_mix", "Flower visits by functional group in each cover crop mix.")],
        2: [("fig", "Fig_effects_forest", "Effects of mix and microclimate on total visitor abundance (incidence rate ratios, 95% CI) in the main model (seasonal trend) and in the sensitivity model (survey date as random effect).", 14),
            ("tab", "T3_anova_total_abundance", "Effects of cover crop mix, microclimate and season on total visitor abundance."),
            ("tab", "T4_anova_total_abundance_date_random", "Sensitivity analysis of total visitor abundance with survey date as a random effect."),
            ("tab", "T5_emmeans_total_abundance", "Model-estimated flower visits per 4-min observation in each cover crop mix.")],
        3: [("fig", "Fig_functional_groups", "(a) Model-estimated visits by pollinators and other flower visitors in each mix; (b) proportion of pollinators among all visitors. Letters: Šidák-adjusted comparisons.", 16),
            ("fig", "Fig_temporal_functional_groups", "Visits per observation (mean ± SE) by functional group across days after sowing.", 16),
            ("tab", "T7_anova_functional_group", "Effects of cover crop mix and functional group on visitor abundance."),
            ("tab", "T8_emmeans_functional_group", "Model-estimated visits per observation by pollinators and other flower visitors in each mix."),
            ("tab", "T9_emmeans_proportion_pollinators", "Proportion of pollinators among all flower visitors in each mix.")],
        4: [("fig", "Fig_taxa_by_mix", "Visits per 4-min observation (mean ± SE) of the main taxa in each mix. Letters: Šidák-adjusted comparisons of model means (a = highest mean). Illustrations identify each taxon.", 16),
            ("tab", "T10a_taxon_occurrence", "Occurrence of the flower-visiting taxa across surveys and blocks."),
            ("tab", "T10_anova_by_taxon", "Effect of cover crop mix on the visits of the main taxa.", lambda d: d[d["term"] == "mix"]),
            ("tab", "T11_emmeans_by_taxon", "Model-estimated visits per observation of the main taxa in each mix.")],
        5: [("fig", "Fig_NMDS_taxa", "NMDS of visitor assemblages per plot (Hellinger, Bray–Curtis). Arrows: taxa significantly correlated with the ordination (envfit, p < 0.05).", 14),
            ("tab", "T12_permanova_all_taxa", "PERMANOVA of flower-visitor composition among cover crop mixes."),
            ("tab", "T13_permanova_Pollinators", "PERMANOVA of pollinator composition among cover crop mixes."),
            ("tab", "T13_permanova_Other_flower_visitors", "PERMANOVA of the composition of other flower visitors among cover crop mixes."),
            ("tab", "T14_permanova_pairwise", "Pairwise PERMANOVA comparisons between cover crop mixes."),
            ("tab", "T16_indval", "Indicator taxa of the cover crop mixes.")],
        6: [("fig", "Fig_iNEXT_size_coverage", "Size-based (a) and coverage-based (b) rarefaction and extrapolation of Hill numbers per mix.", 16),
            ("fig", "Fig_rank_abundance", "Rank–abundance curves per mix.", 14),
            ("tab", "T18_alpha_diversity_tests", "Alpha diversity of flower visitors per plot in each cover crop mix."),
            ("fig", "Fig_diversity_by_group", "Abundance and diversity indices per plot (mean ± SE) in each cover crop mix, for all flower visitors together, pollinators only and other flower visitors only. Letters: Šidák comparisons (a = highest mean); P: effect of mix (linear model with block).", 16),
            ("tab", "T43_alpha_diversity_by_group_tests", "Abundance and alpha diversity per plot of all flower visitors, pollinators and other flower visitors in each cover crop mix."),
            ("tab", "T20_inext_sample_coverage", "Sample size, observed richness and sample coverage of flower visitors in each mix."),
            ("tab", "T21_inext_equal_coverage", "Hill numbers of flower visitors at equal sample coverage.")],
        7: [("fig", "Fig_beta_partition", "Partition of within-mix beta diversity: incidence (turnover + nestedness) and abundance (balanced variation + abundance gradient).", 14),
            ("tab", "T22_beta_partition_within_mix", "Partition of beta diversity among plots within each cover crop mix.")],
        8: [("tab", "T25_temporal_stability", "Temporal variability and taxon synchrony of flower visits in each mix.",
             lambda d: d.groupby("mix", as_index=False)[["cv", "synchrony"]].mean())],
        9: [("fig", "Fig_correlation_diversity_climate", "Spearman correlations between diversity indices and microclimate per mix (observations with visitors). * p_Holm < 0.05, ** < 0.01, *** < 0.001.", 16),
            ("tab", "T27_spearman_diversity_climate_by_mix", "Significant Spearman correlations between flower-visitor indices and microclimate.",
             lambda d: d[(d["p_holm"] < 0.05) | (d["p_partial_holm"] < 0.05)])],
        10: [("fig", "Fig_RDA_climate", "Redundancy analysis of visitor composition per observation constrained by microclimate and mix. Diamonds: mix centroids; black arrows: microclimate; grey arrows: taxa.", 14),
             ("tab", "T28_rda_climate_margin", "Effects of microclimate on flower-visitor composition (partial RDA)."),
             ("tab", "T29_rda_climate_after_season", "Effects of microclimate on flower-visitor composition after accounting for the seasonal trend (partial RDA)."),
             ("tab", "T30_variance_partitioning", "Variation partitioning of flower-visitor composition among mix, microclimate and time.")],
        11: [("fig", "Fig_network", "Mix–visitor network. Line width is proportional to the number of visits.", 14),
             ("tab", "T23_network_metrics", "Network-level metrics of the mix–visitor network."),
             ("tab", "T24_network_taxon_specialization", "Degree and specialisation (d′) of each visitor taxon.")],
        12: [("fig", "Fig_cover_crop_species", "Species composition of the three cover crop mixes (percentage of seed mass). Families other than Poaceae bear insect-visited flowers. Illustrations are schematic.", 14),
             ("tab", "T32_cover_crop_mix_summary", "Plant species composition of the cover crop mixes.")],
        13: [("fig", "Fig_climate_effects_diversity", "Standardised effects (β ± 95% CI) of microclimate on visitor abundance and diversity per observation (linear mixed models with mix, block, seasonal trend and plot). Red: significant negative effect; grey: not significant.", 16),
             ("fig", "Fig_path_model", "Path model linking the seasonal trend, microclimate, visitor abundance and diversity. Only significant paths are shown; values are standardised coefficients.", 16),
             ("tab", "T33_climate_effects_on_diversity", "Standardised effects of microclimate on flower-visitor abundance and diversity.",
              lambda d: d[["response", "term", "beta", "se", "p", "R2_marginal", "LRT_climate_chisq", "LRT_climate_p"]]),
             ("tab", "T34_path_coefficients", "Path coefficients of the piecewise structural equation model."),
             ("tab", "T35_path_model_fit", "Tests of independence claims of the path model."),
             ("tab", "T36_direct_indirect_climate_effects", "Direct and indirect effects of microclimate on flower-visitor richness and diversity.")],
        14: [("fig", "Fig_biomass_diversity", "(a) Fresh biomass of the cover crop mixes (mean and 95% CI; letters: Šidák comparisons, a = highest mean). (b) Visitor abundance, richness, Shannon diversity and proportion of pollinators per plot against biomass.", 16),
             ("tab", "T37_anova_biomass", "Effects of cover crop mix and cropping system on cover crop biomass."),
             ("tab", "T38_emmeans_biomass", "Fresh biomass of the cover crop mixes."),
             ("tab", "T39_biomass_diversity_plot_level", "Relationships between cover crop biomass and flower-visitor indices per plot."),
             ("tab", "T40_abundance_model_with_biomass", "Effects of cover crop biomass on visitor abundance per observation."),
             ("tab", "T41_permanova_biomass", "PERMANOVA of flower-visitor composition with cover crop mix, biomass and cropping system.")],
    }

    doc = Document()
    st = doc.styles["Normal"]; st.font.name = "Arial"; st.font.size = Pt(10)
    n_fig = n_tab = 0

    def inline(par, text, italic=False, size=None):
        # **negrito** e *itálico* do markdown
        for part in re.split(r"(\*\*[^*]+\*\*|\*[^*]+\*)", text):
            if not part: continue
            if part.startswith("**"): r = par.add_run(part[2:-2]); r.bold = True
            elif part.startswith("*"): r = par.add_run(part[1:-1]); r.italic = True
            else: r = par.add_run(part)
            if italic: r.italic = not r.italic if part.startswith("*") and not part.startswith("**") else True
            if size: r.font.size = Pt(size)

    def fmt(v, col):
        if isinstance(v, float):
            if pd.isna(v): return ""
            if re.search(r"(^p$|^p_|Pr\(|p\.value|pval)", col, re.I):
                return "< 0.001" if v < 0.001 else f"{v:.3f}"
            return f"{v:.3g}" if abs(v) >= 1000 or abs(v) < 0.01 else f"{v:.2f}" if abs(v) >= 10 else f"{v:.3f}"
        return str(v)

    tabelas = []
    def add_table(name, caption, transform=None):
        nonlocal n_tab
        path = os.path.join(tabs_dir, name + ".csv")
        if not os.path.exists(path):
            p = doc.add_paragraph(); r = p.add_run(f"[Tabela {name}: gerada ao rodar o notebook no Colab]"); r.italic = True
            return
        tab, note = formatar(name, pd.read_csv(path))
        n_tab += 1
        tabela_word(doc, n_tab, caption, tab, note)
        tabelas.append((caption, tab, note))

    def add_fig(name, caption, width):
        nonlocal n_fig
        path = os.path.join(figs_dir, name + ".png")
        if not os.path.exists(path):
            p = doc.add_paragraph(); r = p.add_run(f"[Figura {name}: gerada ao rodar o notebook no Colab]"); r.italic = True
            return
        n_fig += 1
        doc.add_picture(path, width=Cm(width)); doc.paragraphs[-1].alignment = WD_ALIGN_PARAGRAPH.CENTER
        cap = doc.add_paragraph(); r = cap.add_run(f"Figure {n_fig}. "); r.bold = True; cap.add_run(caption)
        cap.add_run(f" [{name}]").font.color.rgb = RGBColor(0x80, 0x80, 0x80)

    def flush(sec):
        if sec in assets:
            h = doc.add_paragraph(); r = h.add_run("Figuras e tabelas desta seção"); r.bold = True; r.font.color.rgb = RGBColor(0x1F, 0x4E, 0x79)
            for a in assets[sec]:
                if a[0] == "fig": add_fig(a[1], a[2], a[3])
                else: add_table(a[1], a[2], a[3] if len(a) > 3 else None)

    lines = open(md_path, encoding="utf-8").read().splitlines()
    sec, i = None, 0
    while i < len(lines):
        ln = lines[i]
        m = re.match(r"^(#{1,3}) (.*)", ln)
        if m:
            if len(m.group(1)) == 2: flush(sec); ms = re.match(r"(\d+)\.", m.group(2)); sec = int(ms.group(1)) if ms else None
            doc.add_heading(m.group(2), level=len(m.group(1)) - 1 if len(m.group(1)) > 1 else 0)
        elif ln.startswith("|"):
            rows = []
            while i < len(lines) and lines[i].startswith("|"):
                if not re.match(r"^\|[-| :]+\|$", lines[i]): rows.append([c.strip() for c in lines[i].strip("|").split("|")])
                i += 1
            t = doc.add_table(rows=0, cols=len(rows[0])); t.style = "Light Grid Accent 1"
            for rw in rows:
                cells = t.add_row().cells
                for k, c in enumerate(rw): cells[k].text = c
            continue
        elif ln.startswith(">"):
            txt = ln.lstrip("> ").rstrip()
            if txt:
                bm = re.match(r"^- (.*)", txt)
                p = doc.add_paragraph(style="List Bullet" if bm else None)
                p.paragraph_format.left_indent = Cm(1); inline(p, bm.group(1) if bm else txt, italic=False)
                for r in p.runs: r.font.color.rgb = RGBColor(0x1F, 0x3A, 0x5F)
        elif re.match(r"^\s*- ", ln):
            lvl = (len(ln) - len(ln.lstrip())) // 2
            p = doc.add_paragraph(style="List Bullet" if lvl == 0 else "List Bullet 2"); inline(p, ln.strip()[2:])
        elif re.match(r"^\d+\. ", ln):
            p = doc.add_paragraph(style="List Number"); inline(p, re.sub(r"^\d+\. ", "", ln))
        elif ln.strip() in ("", "---"):
            pass
        else:
            inline(doc.add_paragraph(), ln)
        i += 1
    flush(sec)
    doc.save(out_path)
    print(f"Documento salvo: {out_path} | {n_fig} figuras, {n_tab} tabelas")
    if tables_path:   # só as tabelas, prontas para o manuscrito
        dt = Document(); stt = dt.styles["Normal"]; stt.font.name = "Arial"; stt.font.size = Pt(10)
        for k, (cap, tab, note) in enumerate(tabelas, 1):
            tabela_word(dt, k, cap, tab, note)
        dt.save(tables_path); print(f"Tabelas salvas: {tables_path}")

# Rascunho do manuscrito: Material e métodos (Fig. 1), Resultados em português com Fig. 2–8 (legendas em inglês)
# e Material suplementar em inglês (Tables S e Figs. S numeradas pela ordem da primeira citação no texto).
import os, re
import pandas as pd
from docx import Document
from docx.shared import Pt, Cm, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH, WD_COLOR_INDEX

MAIN_FIGS = {
 "Fig_cover_crop_species": (1, "Species composition of the three cover crop mixes (percentage of seed mass). Families other than Poaceae bear insect-visited flowers. Plant illustrations are schematic.", 15),
 "Fig_temporal_functional_groups": (2, "Flower visits by pollinators and other flower visitors in the three cover crop mixes across the flowering season (mean ± SE per 4-min observation; n = 8 plots per mix and survey; 12 surveys, 36–66 days after sowing).", 16.5),
 "Fig_functional_groups": (3, "(a) Visits per 4-min observation by pollinators and other flower visitors in each cover crop mix (model-estimated means and 95% CI; negative binomial GLMM). (b) Proportion of pollinators among all flower visitors (beta-binomial GLMM). Means with the same letter do not differ (Šidák; a = highest mean).", 16.5),
 "Fig_taxa_by_mix": (4, "Visits per 4-min observation (mean ± SE) of the six main flower-visiting taxa in each cover crop mix. Letters: Šidák-adjusted comparisons of model means (a = highest mean). Illustrations are schematic.", 16.5),
 "Fig_network": (5, "Mix–visitor network. Line width is proportional to the number of visits; node size of each mix is proportional to its total visits, and numbers in parentheses are total visits per taxon. Illustrations are schematic.", 14),
 "Fig_NMDS_taxa": (6, "Non-metric multidimensional scaling (NMDS) of flower-visitor assemblages per plot (Hellinger transformation, Bray–Curtis distance; stress = 0.17). Arrows: taxa significantly correlated with the ordination (envfit, P < 0.05).", 13),
 "Fig_diversity_by_group": (7, "Abundance and alpha diversity of flower visitors per plot (mean ± SE) in each cover crop mix, for all flower visitors together, pollinators only and other flower visitors only. Means with the same letter do not differ (Šidák; a = highest mean); P: effect of mix (linear model with block).", 16.5),
 "Fig_path_model": (8, "Path model linking the seasonal trend (days after sowing), microclimate, visitor abundance and diversity per observation. Only significant paths are shown; values are standardised coefficients and arrow width is proportional to effect size. Blue: positive; red: negative; black: seasonal trend. Marginal R² is shown inside each response. Fisher's C = 1.18, df = 6, P = 0.98.", 16.5),
}

SUP_TABS = {
 "T32_cover_crop_mix_summary": "Plant species composition of the cover crop mixes.",
 "T1_abundance_taxon_mix": "Flower visits per taxon in the three cover crop mixes.",
 "T2_abundance_functional_group_mix": "Flower visits by functional group in each cover crop mix.",
 "T10a_taxon_occurrence": "Occurrence of the flower-visiting taxa across surveys and blocks.",
 "T3_anova_total_abundance": "Effects of cover crop mix, microclimate and season on total visitor abundance.",
 "T5_emmeans_total_abundance": "Model-estimated flower visits per 4-min observation in each cover crop mix.",
 "T4_anova_total_abundance_date_random": "Sensitivity analysis of total visitor abundance with survey date as a random effect.",
 "T7_anova_functional_group": "Effects of cover crop mix and functional group on visitor abundance.",
 "T8_emmeans_functional_group": "Model-estimated visits per observation by pollinators and other flower visitors in each mix.",
 "T9_emmeans_proportion_pollinators": "Proportion of pollinators among all flower visitors in each mix.",
 "T10_anova_by_taxon": "Effect of cover crop mix on the visits of the main taxa.",
 "T11_emmeans_by_taxon": "Model-estimated visits per observation of the main taxa in each mix.",
 "T23_network_metrics": "Network-level metrics of the mix–visitor network.",
 "T24_network_taxon_specialization": "Degree and specialisation (d′) of each visitor taxon.",
 "T12_permanova_all_taxa": "PERMANOVA of flower-visitor composition among cover crop mixes.",
 "T14_permanova_pairwise": "Pairwise PERMANOVA comparisons between cover crop mixes.",
 "T13_permanova_Pollinators": "PERMANOVA of pollinator composition among cover crop mixes.",
 "T13_permanova_Other_flower_visitors": "PERMANOVA of the composition of other flower visitors among cover crop mixes.",
 "T16_indval": "Indicator taxa of the cover crop mixes.",
 "T18_alpha_diversity_tests": "Alpha diversity of all flower visitors per plot in each cover crop mix.",
 "T43_alpha_diversity_by_group_tests": "Abundance and alpha diversity per plot of all flower visitors, pollinators and other flower visitors in each cover crop mix.",
 "T20_inext_sample_coverage": "Sample size, observed richness and sample coverage of flower visitors in each mix.",
 "T21_inext_equal_coverage": "Hill numbers of flower visitors at equal sample coverage.",
 "T22_beta_partition_within_mix": "Partition of beta diversity among plots within each cover crop mix.",
 "T25_temporal_stability": "Temporal variability and taxon synchrony of flower visits in each mix.",
 "T35_path_model_fit": "Tests of independence claims of the path model.",
 "T34_path_coefficients": "Path coefficients of the piecewise structural equation model.",
 "T36_direct_indirect_climate_effects": "Direct and indirect effects of microclimate on flower-visitor richness and diversity.",
 "T33_climate_effects_on_diversity": "Standardised effects of microclimate on flower-visitor abundance and diversity.",
 "T27_spearman_diversity_climate_by_mix": "Significant Spearman correlations between flower-visitor indices and microclimate.",
 "T28_rda_climate_margin": "Effects of microclimate on flower-visitor composition (partial RDA).",
 "T29_rda_climate_after_season": "Effects of microclimate on flower-visitor composition after accounting for the seasonal trend (partial RDA).",
 "T30_variance_partitioning": "Variation partitioning of flower-visitor composition among mix, microclimate and time.",
 "T37_anova_biomass": "Effects of cover crop mix and cropping system on cover crop biomass.",
 "T38_emmeans_biomass": "Fresh biomass of the cover crop mixes.",
 "T39_biomass_diversity_plot_level": "Relationships between cover crop biomass and flower-visitor indices per plot.",
 "T40_abundance_model_with_biomass": "Effects of cover crop biomass on visitor abundance per observation.",
 "T41_permanova_biomass": "PERMANOVA of flower-visitor composition with cover crop mix, biomass and cropping system.",
}
SUP_FILTER = {
 "T10_anova_by_taxon": lambda d: d[d["term"] == "mix"],
 "T25_temporal_stability": lambda d: d.groupby("mix", as_index=False)[["cv", "synchrony"]].mean(),
 "T27_spearman_diversity_climate_by_mix": lambda d: d[(d["p_holm"] < 0.05) | (d["p_partial_holm"] < 0.05)],
 "T33_climate_effects_on_diversity": lambda d: d[["response", "term", "beta", "se", "p", "R2_marginal", "LRT_climate_chisq", "LRT_climate_p"]],
}
SUP_FIGS = {
 "Fig_effects_forest": ("Effects of mix and microclimate on total visitor abundance (incidence rate ratios, 95% CI) in the main model (seasonal trend) and in the sensitivity model (survey date as random effect).", 14),
 "Fig_network": ("Mix–visitor network. Line width is proportional to the number of visits; numbers in parentheses are total visits per taxon. Suggested graphical abstract.", 14),
 "Fig_iNEXT_size_coverage": ("Size-based (a) and coverage-based (b) rarefaction and extrapolation of Hill numbers of flower visitors per mix.", 16),
 "Fig_rank_abundance": ("Rank–abundance curves of flower visitors per mix.", 13),
 "Fig_beta_partition": ("Partition of within-mix beta diversity: incidence (turnover + nestedness) and abundance (balanced variation + abundance gradient).", 14),
 "Fig_climate_effects_diversity": ("Standardised effects (β ± 95% CI) of microclimate on visitor abundance and diversity per observation (linear mixed models with mix, block, seasonal trend and plot). Red: significant negative effect; grey: not significant.", 15),
 "Fig_correlation_diversity_climate": ("Spearman correlations between diversity indices and microclimate per mix (observations with visitors). Blue: positive; red: negative. Asterisks: P_Holm < 0.05 (one), < 0.01 (two) and < 0.001 (three).", 16),
 "Fig_RDA_climate": ("Redundancy analysis of flower-visitor composition per observation constrained by microclimate and mix. Diamonds: mix centroids; black arrows: microclimate; grey arrows: taxa.", 13),
 "Fig_biomass_diversity": ("(a) Fresh biomass of the cover crop mixes (mean and 95% CI; letters: Šidák comparisons, a = highest mean). (b) Visitor abundance, richness, Shannon diversity and proportion of pollinators per plot against biomass.", 16),
 "Fig_DHARMa_residuals": ("Residual diagnostics (DHARMa) of the abundance models: uniformity, dispersion and zero-inflation tests.", 15),
}

METODOS = [
 ("h2", "2.X Composição dos mixes de plantas de cobertura"),
 ("p", "Os três mixes diferiram no número de espécies e na proporção de espécies com flores visitadas por insetos (Fig. 1; {T:T32_cover_crop_mix_summary}). O mix Carbon foi composto por seis espécies de duas famílias, com 60% da massa de sementes em gramíneas (*Urochloa brizantha*, *U. dictyoneura*, *Megathyrsus maximus* e *Pennisetum glaucum*); *Crotalaria* sp. e *Vigna unguiculata* foram as únicas espécies com flores visitadas por insetos (40% da massa de sementes). O mix Pollinator também teve seis espécies, de três famílias, das quais três com flores visitadas por insetos (*Helianthus annuus*, *V. unguiculata* e *Crotalaria* sp.; 60% da massa de sementes). O mix Multipurpose, uma mistura comercial, foi o mais diverso, com 11 espécies de cinco famílias, nove delas com flores visitadas por insetos (80% da massa de sementes): três espécies de *Crotalaria*, *Cajanus cajan*, *V. unguiculata*, duas Brassicaceae (*Crambe abyssinica* e *Raphanus sativus*), *H. annuus* e *Fagopyrum esculentum*. As gramíneas (Poaceae) foram consideradas polinizadas pelo vento. [[Conferir: espécie de crotalária dos mixes Carbon e Pollinator e composição do Multipurpose.]]"),
 ("fig", "Fig_cover_crop_species"),
 ("h2", "2.Y Análise dos dados (trecho)"),
 ("p", "A última avaliação (01/10/2025, 74 dias após a semeadura) foi excluída das análises porque coincidiu com o fim da floração dos mixes, quando quase não houve visitas (2,3 visitas por observação, contra 11 a 28 nas demais datas). As análises usaram, portanto, 12 avaliações, de 36 a 66 dias após a semeadura. [[Incluir esta justificativa na seção de análise de dados dos Métodos.]]"),
]

RESULTADOS = [
 ("h1", "3. Resultados"),
 ("h2", "3.1. Comunidade de visitantes florais"),
 ("p", "Nas 12 avaliações (36 a 66 dias após a semeadura), foram registradas 5.740 visitas florais de 12 táxons nas 24 parcelas ({T:T1_abundance_taxon_mix}). Os polinizadores responderam por 87,0% das visitas ({T:T2_abundance_functional_group_mix}). O Apoidea morfotipo 1 foi o visitante dominante (44,5% das visitas), seguido por *Apis mellifera* (18,1%), Apoidea morfotipo 2 (9,4%), Syrphidae (9,1%), *Xylocopa* sp. (8,3%) e *Trigona pallens* (6,6%); os outros seis táxons somaram cerca de 4% das visitas ({T:T10a_taxon_occurrence}). *Bombus* sp., incluído na chave de identificação, não foi registrado. O mix Multipurpose recebeu 2.102 visitas, o Carbon 2.023 e o Pollinator 1.615."),
 ("h2", "3.2. Abundância de polinizadores e de outros visitantes florais"),
 ("p", "A visitação variou ao longo da estação (termo quadrático dos dias após a semeadura: χ² = 9,05; gl = 2; P = 0,011). As visitas de polinizadores aumentaram até o meio da estação, enquanto os outros visitantes florais foram mais frequentes nas primeiras avaliações e diminuíram depois de cerca de 45 dias após a semeadura (Fig. 2)."),
 ("fig", "Fig_temporal_functional_groups"),
 ("p", "A abundância total de visitantes diferiu entre os mixes (χ² = 8,33; gl = 2; P = 0,016; {T:T3_anova_total_abundance}). O mix Pollinator recebeu 22% menos visitas que o Carbon (razão de taxas, IRR = 0,78; IC 95% 0,64–0,95), com médias estimadas de 18,4 visitas por observação de 4 min no Pollinator, 23,6 no Carbon e 23,9 no Multipurpose ({T:T5_emmeans_total_abundance}; {F:Fig_effects_forest}). As visitas diminuíram com a luminosidade (IRR = 0,91 por desvio-padrão; P = 0,006) e tenderam a diminuir com a velocidade do vento (IRR = 0,92; IC 95% 0,85–1,00; P = 0,059), mas não variaram com a temperatura do ar (P = 0,19). Quando a data de avaliação entrou no modelo como efeito aleatório, comparando assim parcelas observadas no mesmo dia, o efeito do mix se manteve (χ² = 7,90; P = 0,019) e o vento reduziu as visitas em 10% a cada desvio-padrão (IRR = 0,90; IC 95% 0,82–0,98; P = 0,012), assim como a luminosidade (IRR = 0,92; P = 0,023) ({T:T4_anova_total_abundance_date_random})."),
 ("p", "O efeito do mix dependeu do grupo funcional (interação mix × grupo funcional: χ² = 27,9; gl = 2; P < 0,001; {T:T7_anova_functional_group}; Fig. 3a). As visitas de polinizadores foram menores no mix Pollinator (14,5 visitas por observação; IC 95% 11,5–18,3) que no Carbon (19,4; 15,4–24,4) e no Multipurpose (22,1; 17,7–27,8). Os outros visitantes florais seguiram padrão diferente: foram menos frequentes no Multipurpose (1,7; 1,1–2,5) que no Carbon (3,8; 2,6–5,7) e no Pollinator (4,0; 2,7–5,9) ({T:T8_emmeans_functional_group}). Nesse modelo, o vento também reduziu as visitas (χ² = 5,38; gl = 1; P = 0,020). A proporção de polinizadores entre os visitantes foi maior no Multipurpose (93%) que no Carbon (89%) e no Pollinator (85%) (χ² = 25,5; gl = 2; P < 0,001; Fig. 3b; {T:T9_emmeans_proportion_pollinators})."),
 ("fig", "Fig_functional_groups"),
 ("h2", "3.3. Respostas dos principais táxons"),
 ("p", "Entre os seis táxons analisados individualmente, o mix afetou os Syrphidae (χ² = 29,9; gl = 2; P < 0,001), que foram cerca de três vezes menos frequentes no Multipurpose (0,48 visitas por observação) que no Carbon (1,34) e no Pollinator (1,38) ({T:T10_anova_by_taxon}; {T:T11_emmeans_by_taxon}; Fig. 4). O Apoidea morfotipo 1 tendeu a ser menos frequente no Pollinator (4,3) que no Multipurpose (6,0) e no Carbon (5,5) (χ² = 5,69; gl = 2; P = 0,058). *Apis mellifera* (P = 0,86), *Xylocopa* sp. (P = 0,80), *Trigona pallens* (P = 0,098) e Coleoptera (P = 0,32) não diferiram entre os mixes, embora *T. pallens* tenha tendido a ser mais frequente no Pollinator (1,4 contra 0,7 visitas por observação no Multipurpose). O vento reduziu as visitas de Syrphidae (IRR = 0,80 por desvio-padrão; χ² = 10,3; P = 0,001) e tendeu a reduzir as de *A. mellifera* (P = 0,070)."),
 ("fig", "Fig_taxa_by_mix"),
 ("p", "A rede mix–visitante mostrou o mesmo padrão (Fig. 5): a maioria dos táxons visitou os três mixes e a especialização foi baixa ({T:T23_network_metrics}; {T:T24_network_taxon_specialization}). O Apoidea morfotipo 1 dominou as interações nos três mixes; *Xylocopa* sp. e o Apoidea morfotipo 2 interagiram principalmente com o Multipurpose, e os Syrphidae e *T. pallens* principalmente com o Carbon e o Pollinator. [[Conferir no Colab: H2′ e d′.]]"),
 ("fig", "Fig_network"),
 ("h2", "3.4. Composição da comunidade"),
 ("p", "A composição da comunidade de visitantes diferiu entre os mixes (PERMANOVA: R² = 0,21; F = 2,80; P = 0,001; {T:T12_permanova_all_taxa}; Fig. 6). O Multipurpose diferiu do Carbon (R² = 0,15; P Holm = 0,006) e do Pollinator (R² = 0,28; P Holm = 0,006), enquanto Carbon e Pollinator não diferiram entre si (P = 0,47; {T:T14_permanova_pairwise}). A dispersão multivariada foi homogênea entre os mixes (betadisper: F = 0,40; P = 0,67), indicando que a diferença reflete mudança na composição, e não na variabilidade entre parcelas. O efeito do mix foi significativo para a assembleia de polinizadores (R² = 0,17; P = 0,008) e marginal para os outros visitantes (R² = 0,19; P = 0,061) ({T:T13_permanova_Pollinators}; {T:T13_permanova_Other_flower_visitors}). Na NMDS (estresse = 0,17), as parcelas do Multipurpose se associaram aos Apoidea morfotipos 1 e 2 e a *Xylocopa* sp., e as do Carbon e do Pollinator a *A. mellifera*, Syrphidae e *T. pallens*. Nenhum táxon foi indicador significativo de um mix após a correção para testes múltiplos ({T:T16_indval}). [[Conferir no Colab: IndVal com 9999 permutações.]]"),
 ("fig", "Fig_NMDS_taxa"),
 ("h2", "3.5. Diversidade de visitantes"),
 ("p", "Com todos os visitantes juntos, nenhum índice de diversidade diferiu entre os mixes: riqueza (q = 0; 8,1–8,6 táxons por parcela; P = 0,61), diversidade de Shannon (q = 1; P = 0,24), diversidade de Simpson (q = 2; P = 0,29) e equabilidade de Pielou (P = 0,29) (Fig. 7; {T:T18_alpha_diversity_tests}). Separando os grupos, a diversidade de polinizadores também foi semelhante entre os mixes (riqueza de 4,4 a 4,8 táxons; P ≥ 0,46 para todos os índices), e a abundância de polinizadores por parcela tendeu a ser menor no Pollinator (163 visitas, contra 216 no Carbon e 245 no Multipurpose; F = 3,40; P = 0,056). Os outros visitantes florais, com metade da abundância no Multipurpose (18 visitas por parcela, contra 37 no Carbon e 39 no Pollinator; F = 13,0; P < 0,001), formaram nesse mix uma assembleia mais equilibrada (equabilidade de Pielou de 0,70, contra 0,48 no Carbon e 0,53 no Pollinator; F = 6,33; P = 0,008), com maior diversidade de Simpson (2,2 contra 1,5–1,7; P = 0,045) e tendência semelhante para a diversidade de Shannon (P = 0,055) (Fig. 7; {T:T43_alpha_diversity_by_group_tests})."),
 ("fig", "Fig_diversity_by_group"),
 ("p", "As curvas de rarefação e extrapolação indicaram cobertura amostral próxima de 100% nos três mixes ({T:T20_inext_sample_coverage}; {T:T21_inext_equal_coverage}; {F:Fig_iNEXT_size_coverage}; {F:Fig_rank_abundance}). [[Conferir no Colab: valores de cobertura e diversidade na mesma cobertura (iNEXT).]] A diversidade beta entre parcelas do mesmo mix foi numericamente menor no Multipurpose (Bray–Curtis 0,26, contra 0,35 no Carbon e 0,31 no Pollinator), mas não diferiu entre os mixes (betadisper: F = 1,80; P = 0,19; {T:T22_beta_partition_within_mix}; {F:Fig_beta_partition}). A variabilidade temporal da visitação também não diferiu (coeficiente de variação de 0,52 a 0,69; P = 0,41), e a sincronia entre táxons foi baixa nos três mixes (0,12 a 0,18; P = 0,44; {T:T25_temporal_stability})."),
 ("h2", "3.6. Efeitos do microclima sobre a abundância e a diversidade"),
 ("p", "O modelo de caminhos se ajustou bem aos dados (C de Fisher = 1,18; gl = 6; P = 0,98; {T:T35_path_model_fit}) e explicou 22% da variação na abundância, 27% na riqueza e 30% na diversidade de Shannon (R² marginal; Fig. 8; {T:T34_path_coefficients}). A tendência sazonal determinou o microclima: o vento diminuiu e depois se estabilizou ao longo da estação (β linear = −0,34; β quadrático = 0,35; P < 0,001), e a temperatura diminuiu (β = −0,30; P < 0,001). O vento (β = −0,14; P = 0,031) e a luminosidade (β = −0,14; P = 0,015) reduziram a abundância de visitantes, e a abundância aumentou a riqueza (β = 0,18; P = 0,001). O microclima não teve efeito direto sobre a riqueza nem sobre a diversidade de Shannon (P ≥ 0,11); o efeito do vento sobre a riqueza ocorreu por meio da abundância (efeito indireto β = −0,03; P = 0,074; {T:T36_direct_indirect_climate_effects})."),
 ("fig", "Fig_path_model"),
 ("p", "Os modelos mistos de cada índice confirmaram que o microclima afetou em conjunto a abundância (χ² = 11,2; gl = 3; P = 0,011) e a riqueza (χ² = 8,31; gl = 3; P = 0,040), mas não as diversidades de Shannon (P = 0,36) e de Simpson (P = 0,39), a equabilidade (P = 0,45) nem a proporção de polinizadores (P = 0,75) ({T:T33_climate_effects_on_diversity}; {F:Fig_climate_effects_diversity}). O vento reduziu a abundância (β = −0,14; IC 95% −0,27 a −0,01) e a riqueza (β = −0,13; −0,26 a −0,002), e a luminosidade reduziu a abundância (β = −0,14; −0,25 a −0,03). As correlações de Spearman mostraram o mesmo padrão: a abundância diminuiu com o vento (ρ = −0,24; P Holm = 0,001), também após descontar a tendência sazonal (ρ parcial = −0,20; P Holm = 0,013), e a riqueza e as diversidades de Hill diminuíram com o vento depois de descontada a estação (ρ parcial = −0,22 a −0,27; P Holm ≤ 0,004) ({T:T27_spearman_diversity_climate_by_mix}; {F:Fig_correlation_diversity_climate}). Na RDA parcial (R² ajustado = 0,071), o vento (F = 12,6; P = 0,001) e a temperatura (F = 4,53; P = 0,001) estruturaram a composição das visitas em cada observação; após descontar o mix e a tendência sazonal, a temperatura (F = 7,10; P = 0,001) e a luminosidade (F = 3,05; P = 0,007) continuaram significativas, mas o vento não (P = 0,18) ({T:T28_rda_climate_margin}; {T:T29_rda_climate_after_season}; {F:Fig_RDA_climate}). Na partição da variância, o tempo (dias após a semeadura) teve a maior fração exclusiva da variação na composição (19,4%), contra 1,8% do microclima e 1,4% do mix ({T:T30_variance_partitioning})."),
 ("h2", "3.7. Biomassa das plantas de cobertura"),
 ("p", "A biomassa fresca diferiu entre os mixes (F = 4,40; gl = 2; P = 0,029; {T:T37_anova_biomass}; {F:Fig_biomass_diversity}): foi maior no Pollinator (30,8 Mg ha⁻¹) que no Carbon (20,2 Mg ha⁻¹), com o Multipurpose intermediário (23,9 Mg ha⁻¹; {T:T38_emmeans_biomass}). O sistema de cultivo (soja ou rotação) não afetou a biomassa (P = 0,70). A biomassa não se relacionou com o total de visitas, a riqueza, a diversidade de Shannon nem a proporção de polinizadores por parcela (P ≥ 0,15; {T:T39_biomass_diversity_plot_level}), nem com as visitas por observação (P = 0,39; {T:T40_abundance_model_with_biomass}). Dentro dos mixes, parcelas com mais biomassa tiveram diversidade de Simpson (P = 0,019) e equabilidade (P = 0,033) um pouco menores. A biomassa não explicou a composição após descontar o mix (PERMANOVA: R² = 0,07; P = 0,10), enquanto o mix permaneceu significativo (R² = 0,26; P < 0,001; {T:T41_permanova_biomass})."),
]


def gerar_manuscrito(figs_dir, tabs_dir, out_path):
    # Numeração do suplementar pela ordem da primeira citação
    texto = " ".join(c for k, c in METODOS + RESULTADOS if k == "p")
    num_t, num_f = {}, {}
    for kind, name in re.findall(r"\{([TF]):([^}]+)\}", texto):
        d = num_t if kind == "T" else num_f
        if name not in d: d[name] = len(d) + 1
    num_f.setdefault("Fig_DHARMa_residuals", len(num_f) + 1)

    doc = Document()
    st = doc.styles["Normal"]; st.font.name = "Times New Roman"; st.font.size = Pt(12)
    st.paragraph_format.line_spacing = 2.0; st.paragraph_format.space_after = Pt(0)
    for s in ("Heading 1", "Heading 2", "Title"):
        doc.styles[s].font.name = "Times New Roman"; doc.styles[s].font.color.rgb = RGBColor(0, 0, 0)
    doc.styles["Heading 1"].font.size = Pt(13); doc.styles["Heading 2"].font.size = Pt(12)

    def runs(par, text, size=None):
        for part in re.split(r"(\[\[.+?\]\]|\*[^*]+\*)", text):
            if not part: continue
            if part.startswith("[["):
                r = par.add_run(part[2:-2]); r.font.highlight_color = WD_COLOR_INDEX.YELLOW
            elif part.startswith("*"):
                r = par.add_run(part[1:-1]); r.italic = True
            else:
                r = par.add_run(part)
            if size: r.font.size = Pt(size)

    def ref(m):
        kind, name = m.group(1), m.group(2)
        return f"Table S{num_t[name]}" if kind == "T" else f"Fig. S{num_f[name]}"

    def legenda(par, label, caption, size=10):
        par.paragraph_format.line_spacing = 1.15
        r = par.add_run(label + " "); r.bold = True; r.font.size = Pt(size)
        runs(par, caption, size)

    def figura(path, width, label, caption):
        if os.path.exists(path):
            doc.add_picture(path, width=Cm(width)); doc.paragraphs[-1].alignment = WD_ALIGN_PARAGRAPH.CENTER
            doc.paragraphs[-1].paragraph_format.line_spacing = 1.0
        else:
            p = doc.add_paragraph(); runs(p, f"[[Figura gerada ao rodar o notebook no Colab ({os.path.basename(path)}).]]")
        p = doc.add_paragraph(); legenda(p, label, caption); p.paragraph_format.space_after = Pt(12)

    def bloco(itens):
        for kind, c in itens:
            if kind == "h1": doc.add_heading(c, level=1)
            elif kind == "h2": doc.add_heading(c, level=2)
            elif kind == "p":
                p = doc.add_paragraph(); p.paragraph_format.first_line_indent = Cm(1.0)
                runs(p, re.sub(r"\{([TF]):([^}]+)\}", ref, c))
            elif kind == "fig":
                n, cap, w = MAIN_FIGS[c]
                figura(os.path.join(figs_dir, c + ".png"), w, f"Fig. {n}.", cap)

    t = doc.add_paragraph(); r = t.add_run("Rascunho do manuscrito – Material e métodos (composição dos mixes), Resultados e Material suplementar"); r.bold = True; r.font.size = Pt(14)
    p = doc.add_paragraph(); p.paragraph_format.line_spacing = 1.15
    runs(p, "Texto dos Resultados em português; legendas das figuras e material suplementar em inglês (Agriculture, Ecosystems & Environment). Todas as tabelas estão no material suplementar, numeradas pela ordem de citação. Os trechos em amarelo precisam de conferência. Os nomes dos mixes seguem os das figuras (Carbon, Multipurpose e Pollinator).", 10)

    doc.add_heading("2. Material e métodos (trecho)", level=1)
    bloco(METODOS)
    bloco(RESULTADOS)

    doc.add_page_break()
    doc.add_heading("Supplementary material", level=1)
    p = doc.add_paragraph(); runs(p, "Appendix A. Supplementary data for: [manuscript title]. Tables S1–S%d and Figs. S1–S%d." % (len(num_t), len(num_f)), 11)
    doc.add_heading("Supplementary tables", level=2)
    for name, n in sorted(num_t.items(), key=lambda x: x[1]):
        path = os.path.join(tabs_dir, name + ".csv")
        if not os.path.exists(path):
            p = doc.add_paragraph(); legenda(p, f"Table S{n}.", SUP_TABS[name])
            p = doc.add_paragraph(); runs(p, f"[[Table generated when the notebook runs in Colab ({name}.csv).]]", 10)
            continue
        d = pd.read_csv(path)
        if name in SUP_FILTER: d = SUP_FILTER[name](d)
        tab, note = formatar(name, d)
        tabela_word(doc, f"S{n}", SUP_TABS[name], tab, note)
    doc.add_page_break()
    doc.add_heading("Supplementary figures", level=2)
    for name, n in sorted(num_f.items(), key=lambda x: x[1]):
        cap, w = SUP_FIGS[name]
        figura(os.path.join(figs_dir, name + ".png"), w, f"Fig. S{n}.", cap)
    doc.add_heading("Appendix B. Identification key to flower-visitor morphogroups", level=2)
    p = doc.add_paragraph(); runs(p, "[[Insert the identification key used in the field (morphogroups, diagnostic characters and photographs).]]", 11)

    for tb in doc.tables:   # tabelas em espaço simples
        for row in tb.rows:
            for cell in row.cells:
                for par in cell.paragraphs: par.paragraph_format.line_spacing = 1.0
    for par in doc.paragraphs:
        if par.runs and par.runs[0].text.startswith("Table S"): par.paragraph_format.line_spacing = 1.15
    doc.save(out_path)
    print("Manuscrito salvo:", out_path, "|", len(MAIN_FIGS), "figuras no texto,", len(num_t), "tabelas S,", len(num_f), "figuras S")
PASTA_MANUSCRITO = "1_fCxQSKGMVQrKXuk1rxip8OUO9oxW-W3"   # 01_Manuscrito
DOCX = "application/vnd.openxmlformats-officedocument.wordprocessingml.document"
baixar(PASTA_MANUSCRITO, "resultados_estatisticos_v2.md", "/content/resultados_estatisticos_v2.md")
saida = "/content/out/Resultados_estatisticos_figuras_tabelas_v2.docx"
saida_tab = "/content/out/Tables_AGEE_v2.docx"
gerar_relatorio("/content/resultados_estatisticos_v2.md", "/content/out/figs", "/content/out/tables", saida, saida_tab)

# Substitui os Google Docs anteriores (vão para a lixeira) e cria os novos, convertidos do .docx
def publicar_doc(arquivo, nome):
    for f in listar(PASTA_MANUSCRITO):
        if f["name"] == nome and f["mimeType"] == "application/vnd.google-apps.document":
            executar(drive.files().update(fileId=f["id"], body={"trashed": True}, **KW))
    media = MediaFileUpload(arquivo, mimetype=DOCX, resumable=True)
    novo = executar(drive.files().create(body={"name": nome, "parents": [PASTA_MANUSCRITO], "mimeType": "application/vnd.google-apps.document"},
                                         media_body=media, fields="id, webViewLink", **KW))
    print(nome, "->", novo["webViewLink"])

publicar_doc(saida, "Resultados estatísticos – guia para a seção Results (v2)")
publicar_doc(saida_tab, "Tables – AGEE format (v2)")

# Rascunho do manuscrito: Resultados em português com Fig. 1–7 e material suplementar em inglês
saida_ms = "/content/out/Manuscrito_resultados_suplementar_v1.docx"
gerar_manuscrito("/content/out/figs", "/content/out/tables", saida_ms)
publicar_doc(saida_ms, "Manuscrito – Resultados e material suplementar (v1)")